# AI-Powered Grievance Classification System — Bengaluru
### Automatically route citizen complaints to the correct civic authority and predict their urgency

---

**Project Overview**

Bengaluru receives thousands of citizen grievances daily across multiple civic agencies — BBMP, BWSSB, BESCOM, BTP, and others. Manually triaging these complaints is slow, error-prone, and costly.

This notebook builds an end-to-end NLP pipeline that:
1. **Routes complaints** to the correct civic authority (civic agency classification)
2. **Prioritises complaints** by predicted urgency: Low → Medium → High → Critical (severity classification)

Both tasks use the same complaint text as input. We benchmark classical ML models (Logistic Regression, LinearSVC, Random Forest, Multinomial Naive Bayes) against deep learning approaches (DistilBERT fine-tuning, BiLSTM).

---

**Notebook Structure**

| Section | Description |
|---------|-------------|
| 1 | Data Retrieval & Cleaning |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Civic Agency Classification — Preprocessing, Augmentation & Training |
| 4 | Civic Agency Classification — Final Dataset & Model Saving |
| 5 | Severity Classification — Preprocessing, Augmentation, Training & Inference |

---


## 1  Data Retrieval & Cleaning

### 1.1  Imports & Environment Setup

All standard libraries, ML frameworks, and NLP tools are loaded here.  
GPU memory is capped at 75 % to prevent OOM errors during augmentation and fine-tuning.


In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import re, copy, json, glob, gzip, random, shutil, logging, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path

# ── Data Manipulation & Visualisation ────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import json
import requests
import time

# ── Environment & Database ───────────────────────────────────────────────────
from dotenv import load_dotenv
from sqlalchemy import create_engine

# ── NLP Tools ────────────────────────────────────────────────────────────────
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.sentence as nas

# Prevent accidental downloads inside a controlled environment
nltk.download = lambda *args, **kwargs: True

# ── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import StratifiedKFold, ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, precision_recall_fscore_support,
    classification_report, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_class_weight

# ── PyTorch & Hugging Face ────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.75, device=0)
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ── Scipy ─────────────────────────────────────────────────────────────────────
from scipy.special import softmax
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu
from scipy.stats import skew

# -- Open AI API --------------------------------------------------------------
from openai import OpenAI
import google.generativeai as genai
import vertexai
from vertexai.generative_models import GenerativeModel

# ── Project Root ─────────────────────────────────────────────────────────────
# Assumes this notebook lives one level inside the project (e.g. /notebook/)
PROJECT_ROOT = Path.cwd().parent
CHARTS_DIR   = PROJECT_ROOT / "charts_and_graphs"
CHARTS_DIR.mkdir(exist_ok=True)
print(f"Project root : {PROJECT_ROOT}")
print(f"Charts folder: {CHARTS_DIR}")


### 1.2  Database Connection

Credentials are stored in a `.env` file at `<project_root>/src/.env` — **never hard-coded**.


In [ ]:
load_dotenv(dotenv_path=PROJECT_ROOT / "src" / ".env", override=True)

DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('user')}:{os.getenv('password')}"
    f"@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as conn:
        print("✅ Database connection successful.")
except Exception as e:
    print(f"❌ Connection failed: {e}")

### 1.3  Load Full Dataset


In [ ]:
df = pd.read_sql("SELECT * FROM bbmc_final_data;", engine)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.columns

In [ ]:
df= df[['created_at','description','civic_agency_id', 'civic_agency_title', 
        'complaints_length', 'severity_score', 
        'confidence_score', 'severity_reason']]

### 1.4  Missing-Value Analysis

We check the percentage of nulls across all columns before deciding which rows to drop.


In [ ]:
df_null_pct = df.isna().mean() * 100
print("Null percentage per column (non-zero only):")
print(df_null_pct[df_null_pct > 0].sort_values(ascending=False))

In [ ]:
df= df.drop(columns=['civic_agency_id'])
df.columns

### 1.5  Drop Null Records

Both `description` (the complaint text) and `severity` (the target label) are mandatory.
Any row missing either field is removed.


In [ ]:
print(f"Rows before cleaning : {df.shape[0]:,}")
df.dropna(subset=["description", "civic_agency_title", "severity_score"], inplace=True)
print(f"Rows after dropping nulls : {df.shape[0]:,}")

### 1.6  Standardise Civic Agency Names

Some agencies appear under both their full name and acronym (e.g. *BBMP* and *Bruhat Bengaluru Mahanagara Palike*), causing artificially split complaint counts.  
We consolidate all variants into a single canonical acronym, and merge very sparse agencies into the closest parent to reduce label sparsity.


In [ ]:
df['civic_agency_title'].value_counts()

In [ ]:
# ── Step 1: full name → acronym ───────────────────────────────────────────────
agency_alias_map = {
    "Bruhat Bengaluru Mahanagara Palike":       "BBMP",
    "Bangalore Traffic Police":                 "BTP",
    "Bangalore Water Supply And Sewerage Board": "BWSSB",
    "Karnataka State Pollution Control Board":  "KSPCB",
    "Bangalore Electricity Supply Company":     "BESCOM",
}
df["civic_agency_title"] = df["civic_agency_title"].replace(agency_alias_map)

# ── Step 2: merge sparse agencies into parent categories ─────────────────────
agency_consolidation_map = {
    "BDA":   "BBMP",       # urban infrastructure overlap
    "BMTC":  "Transport",  # public transport
    "KSRTC": "Transport",
}
df["civic_agency_title"] = df["civic_agency_title"].replace(agency_consolidation_map)

print(f"Unique civic agencies after consolidation: {df['civic_agency_title'].nunique()}")
print(df["civic_agency_title"].value_counts())

### 1.7  Final Deduplication & Index Reset


In [ ]:
df = (
    df.dropna(subset=["description", "severity_score"])
      .drop_duplicates()
      .reset_index(drop=True)
)
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(2)

---
## 2  Exploratory Data Analysis (EDA)

### 2.1  Complaint Length Distribution

We measure the word count of each complaint and examine how it varies across severity categories.  
Very short complaints (fewer than ~14 words) may lack enough context to reliably predict severity — this informs our augmentation threshold.


In [ ]:
PATH_PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

df.to_csv(PATH_PROCESSED_DATA / "bbmc_data_v2.csv", index=False)

df.head(2)

Bucketing the Severity:

In [ ]:
def get_severity(score):
    if score >= 90:
        return "Critical"
    elif score >= 80:
        return "High"
    elif score >= 50:
        return "Medium"
    elif score >= 1:
        return "Low"
    else:
        return "Non-Grievance"

df["severity"] = df["severity_score"].apply(get_severity)

In [ ]:
# Sort chronologically for time-based analysis later
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df = df.sort_values("created_at", ascending=True).reset_index(drop=True)

# Word-count feature
df["complaint_length"] = df["description"].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
sns.histplot(df["complaint_length"], bins=100, kde=True, ax=axes[0])
axes[0].axvline(df["complaint_length"].mean(),   linestyle="--",
                label=f'Mean ({df["complaint_length"].mean():.1f})')
axes[0].axvline(df["complaint_length"].median(), linestyle="-",
                label=f'Median ({df["complaint_length"].median():.0f})')
axes[0].set_title("Overall Complaint Word-Length Distribution")
axes[0].set_xlabel("Word Count"); axes[0].set_ylabel("Frequency")
axes[0].legend()

# By severity
sns.boxplot(x="severity", y="complaint_length", data=df,
            order=["Low", "Medium", "High", "Critical"], ax=axes[1])
axes[1].set_title("Complaint Length by Severity Category")
axes[1].set_xlabel("severity"); axes[1].set_ylabel("Word Count")

plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.1a_complaint_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Descriptive statistics per severity
print("Complaint length statistics by severity:\n")
print(df.groupby("severity")["complaint_length"].describe().round(2))

# KDE overlay
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="complaint_length", hue="severity", fill=True)
plt.title("Complaint Length Density by Severity")
plt.xlabel("Word Count")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.1b_complaint_length_kde.png", dpi=150, bbox_inches="tight")
plt.show()

Detecting the outlier based on the complaint length (unusually short or long complaints):

In [ ]:
def detect_outlier(feature):
    Q1 = feature.quantile(0.25)
    Q3 = feature.quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return lower_bound, upper_bound

In [ ]:
lower_bound, upper_bound = detect_outlier(df["complaints_length"])

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

As the dataset is highly skewed, this si why the lowerbound is apperaing to be negative, lets run a test for detecting the skewness of the tranformation figures and then decide a tranformation based on that:

In [ ]:
print("Original:", skew(df["complaints_length"]))
print("Log:", skew(np.log1p(df["complaints_length"])))
print("Sqrt:", skew(np.sqrt(df["complaints_length"])))
print("Cube Root:", skew(np.cbrt(df["complaints_length"])))

In [ ]:
log_length = np.log1p(df["complaints_length"])

Q1 = log_length.quantile(0.25)
Q3 = log_length.quantile(0.75)

IQR = Q3 - Q1

lower_log = Q1 - 1.5 * IQR
upper_log = Q3 + 1.5 * IQR

print("Log Lower Bound:", lower_log)
print("Log Upper Bound:", upper_log)

In [ ]:
lower_original = np.expm1(lower_log)
upper_original = np.expm1(upper_log)

print("Original Lower Bound:", lower_original)
print("Original Upper Bound:", upper_original)

So as per the log transformed ersults, the lowerbound is 1, so that means we can discard the complaints having length of 0 words that contains only special charcters or symbols, as tehy carry no menaingful conext.

In [ ]:
# Calculate log-transformed complaint length (using singular column name)
log_length = np.log1p(df["complaint_length"])

# Generate probability plot
stats.probplot(log_length, dist="norm", plot=plt)

plt.title("2.1c Q-Q Plot of log1p(complaint_length)")
plt.tight_layout()

# Save figure (ensuring singular name and CHARTS_DIR exists)
plt.savefig(
    CHARTS_DIR / "2.1c_Q-Q Plot of log1p(complaint_length).png", 
    dpi=150, 
    bbox_inches="tight"
)
plt.show()


A comparison of orginal vs log tranformed distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot original distribution (using singular column name)
ax[0].hist(df["complaint_length"], bins=50, color="blue", edgecolor="white")
ax[0].set_title("Original Distribution")
ax[0].set_xlabel("Complaint Length (Words)")
ax[0].set_ylabel("Frequency")

# Plot transformed distribution
ax[1].hist(log_length, bins=50, color="blue", edgecolor="white")
ax[1].set_title("After Log1p Transformation")
ax[1].set_xlabel("log1p(Complaint Length)")
ax[1].set_ylabel("Frequency")

fig.suptitle(
    "Effect of Log Transformation on Complaint Length",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

# Save the figure to your charts directory
plt.savefig(
    CHARTS_DIR / "2.1d_complaint_length_kde.png", 
    dpi=150, 
    bbox_inches="tight"
)

plt.show()


Based on the data and the explainibility we will keep the original numbers as criteria for detecting the too long or too short complaints. So as per that the lower bound is 0 but we will keep that as 1 as 0 complaint length has no useful context and the upper bound is 108 for flagging a complaint length as an outlier.

In [ ]:
# Inspect the shortest complaints — these are candidates for augmentation
df_short = df[df["complaint_length"] < 108][["description", "complaint_length", "severity"]].head(10)
print("Sample complaints under 110 words:")
display(df_short)

In [ ]:
# Percentile breakdown — useful for choosing the 256-word summarisation threshold
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_stats = (
    df.groupby("severity")["complaint_length"]
      .quantile(percentiles)
      .unstack()
      .round(2)
)
pct_stats.columns = [f"{int(p*100)}%" for p in percentiles]
print("Complaint length percentiles by severity:\n")
print(pct_stats)

In [ ]:
df["complaints_length"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999]
)

So based on our analysis 95% complaints are below complaint_length of 108.

### 2.2  Grievance Volume Over Time

We look at how total complaints — broken down by severity — have changed year over year.


In [ ]:
df["year"] = df["created_at"].dt.year

palette = {
    "Non-Grievance": "grey",
    "Low": "green",
    "Medium": "orange",
    "High": "red",
    "Critical": "blue"
}

plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    x="year",
    hue="severity",
    palette=palette
)
plt.title("Number of Grievances by Severity per Year")
plt.xlabel("Year"); plt.ylabel("Complaint Count")
plt.legend(title="Severity")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.2_grievances_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3  Complaint Distribution per Civic Agency

Examining complaint volume across agencies, overall and by year, helps identify disproportionately burdened agencies and informs routing model complexity.


In [ ]:
print(f"Total civic agencies (after consolidation): {df['civic_agency_title'].nunique()}")
print("\nComplaint count per agency:")
print(df["civic_agency_title"].value_counts())

In [ ]:
palette = {
    "Non-Grievance": "grey",
    "Low": "green",
    "Medium": "orange",
    "High": "red",
    "Critical": "blue"
}
# Year-by-year severity breakdown per agency (percentage)
for agency in df["civic_agency_title"].dropna().unique():
    data  = df[df["civic_agency_title"] == agency]
    pivot = data.pivot_table(index="year", columns="severity", aggfunc="size", fill_value=0)
    if pivot.empty:
        continue
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    pivot_pct.plot(kind="bar", stacked=False, figsize=(8, 4), color=[palette[col] for col in pivot_pct.columns])
    plt.ylim(0, 100)
    plt.title(f"{agency} — Complaint Severity Distribution (%) by Year")
    plt.xlabel("Year"); plt.ylabel("% of Complaints")
    plt.legend(title="Severity")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"2.3_{agency}_severity_by_year.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 3  Civic Agency Classification — Preprocessing, Augmentation & Training

**Objective:** Given the free-text description of a complaint, predict which civic agency (*BBMP, BWSSB, BESCOM, BTP, Transport, …*) should handle it.

**Training strategy:** 5-fold stratified cross-validation. Each training fold is augmented independently before fitting; the model is evaluated on the untouched validation fold to prevent data leakage.


In [ ]:
df= df[['description','civic_agency_title','severity','severity_score','complaint_length']]
df.head()

### 3.1  Text Preprocessing

For classical ML models each complaint passes through:
1. Lowercasing
2. Boilerplate removal (greetings, sign-offs)
3. URL removal
4. Non-alphabetic character removal
5. Whitespace normalisation
6. Stopword removal + Lemmatisation


In [ ]:
# ── NLTK setup ────────────────────────────────────────────────────────────────
local_nltk_path = PROJECT_ROOT / "data" / "nltk"
local_nltk_path.mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(local_nltk_path))

try:
    STOPWORDS   = set(stopwords.words("english"))
    _lemmatizer = WordNetLemmatizer()
    print("✅ NLTK stopwords and lemmatizer loaded.")
except LookupError:
    raise RuntimeError(f"❌ NLTK stopwords not found. Check: {local_nltk_path}")

nltk.download = lambda *args, **kwargs: True
logging.getLogger("nltk").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

_BOILERPLATE = [
    r"dear sir.*?", r"dear madam.*?",
    r"regards.*?",  r"sent from my.*?", r"thank you.*?",
]

def preprocess_classical_ml(text: str) -> str:
    text = str(text).lower()
    for pat in _BOILERPLATE:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"http[s]?://\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text  # No lemmatization

# Sanity check
sample = "Dear Sir, The road near my house has deep potholes. Regards, Citizen"
print(f"Before : {sample}")
print(f"After  : {preprocess_classical_ml(sample)}")

Though for NLP based BERT models:

For BERT-based models, each complaint passes through:

1. URL removal
2. Removal of complaint/ticket/reference IDs (if present)
3. Removal of obvious boilerplate metadata (e.g., "Sent from my iPhone")
4. Whitespace normalization
5. Preservation of original casing, punctuation, stopwords, and word forms to retain contextual    information for the BERT tokenizer

In [ ]:
_BOILERPLATE = [
    r"sent from my.*?",
]

def preprocess_nlp_bert(text: str) -> str:
    text = str(text)

    # Remove URLs
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)

    # Remove complaint/ticket/reference IDs
    text = re.sub(
        r"(complaint|ticket|reference)\s*(id|number|no)?\s*[:\-]?\s*\w+",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove boilerplate signatures
    for pat in _BOILERPLATE:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Sanity check
sample = "Dear Sir, The road near my house has deep potholes. Regards, Citizen"
print(f"Before : {sample}")
print(f"After  : {preprocess_nlp_bert(sample)}")

### 3.3  N-gram Frequency Analysis & Word Cloud

Inspecting the top unigrams, bigrams, and trigrams confirms the preprocessed text retains domain-relevant vocabulary (e.g. *road, garbage, water, drainage*) and reveals any remaining noise.


In [ ]:
# Apply preprocessing to the full corpus for EDA
df["clean"] = df["description"].apply(preprocess_classical_ml)

def get_ngram_freq(text_series, ngram_range=(2, 2), top_k=20, min_df=10):
    if text_series.empty:
        return pd.DataFrame(columns=["ngram", "frequency"])
    vec = CountVectorizer(ngram_range=ngram_range, min_df=min_df, stop_words="english")
    X_v = vec.fit_transform(text_series)
    counts = X_v.sum(axis=0).A1
    return (
        pd.DataFrame({"ngram": vec.get_feature_names_out(), "frequency": counts})
          .sort_values("frequency", ascending=False).head(top_k)
    )

def plot_ngram(df_ngram, title, prefix):
    if df_ngram.empty:
        print(f"No data for: {title}"); return
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    ax = sns.barplot(x="frequency", y="ngram", data=df_ngram, palette="viridis")
    plt.title(title, fontsize=14, loc="left")
    plt.xlabel("Frequency"); plt.ylabel("")
    
    for container in ax.containers:
        ax.bar_label(container, padding=3)
        
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    
    # Save fig using prefix (e.g., 3.3a_Top_Unigrams.png)
    filename = f"{prefix}_{title.replace(' ', '_')}.png"
    plt.savefig(CHARTS_DIR / filename, dpi=150, bbox_inches="tight")
    
    plt.show()
    plt.close()

# Run for Unigrams (3.3a), Bigrams (3.3b), and Trigrams (3.3c)
ngrams_config = [
    ((1, 1), "Top Unigrams", "3.3a"),
    ((2, 2), "Top Bigrams", "3.3b"),
    ((3, 3), "Top Trigrams", "3.3c")
]

for ngram_range, title, prefix in ngrams_config:
    df_ngram = get_ngram_freq(df["clean"], ngram_range=ngram_range)
    plot_ngram(df_ngram, title, prefix)


In [ ]:
all_text = " ".join(df["clean"])
wc = WordCloud(width=1000, height=500, background_color="white",
               max_words=200, colormap="viridis").generate(all_text)
plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud — Preprocessed Complaint Corpus", fontsize=16)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "3.2_word_cloud.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4  Data Augmentation Setup

Class imbalance causes under-represented agencies to be poorly classified. Three complementary strategies are used:

- **Contextual word substitution** (DistilBERT) — replaces words with semantically similar alternatives
- **Synonym substitution** (WordNet) — lightweight and deterministic
- **Spelling augmentation** — injects minor typo-style noise for robustness

Complaints ≥ 256 words are **summarised** with T5 instead, as word-level substitution on very long texts adds little diversity.

**Augmentation intensity scales with class size:**

| Class size | Multiplier |
|-----------|-----------|
| > 2000 | skip |
| 1001–2000 | ×1 |
| 501–1000 | ×2 |
| 1-500 | ×3 |


### 3.5  Preprocessing & Augmentation — Civic Agency Folds


In [ ]:
# # ── Load environment variables from src/.env ──────────────────────────────────
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
# load_dotenv(dotenv_path=PROJECT_ROOT / "src" / ".env", override=True)

# vertexai.init(project="ai-grievance-sandi", location="us-central1")
# _gemini = GenerativeModel("gemini-2.5-flash")
# print("[OK] Gemini client initialised via Vertex AI (project: ai-grievance-sandi)")


# # ── Gemini Prompts ────────────────────────────────────────────────────────────
# _GEMINI_LONG_PROMPT = (
#     "Summarise the following civic complaint and write another variant of the complaint. "
#     "Preserve the original meaning, sentiment, and complaint intent. "
#     "Do NOT change or affect any severity metrics/scores mentioned or implied; keep the core issue identical.\n"
#     "Generate {n} different natural-language variations.\n\n"
#     "Return ONLY a valid JSON array of strings and nothing else.\n\n"
#     "Example output format:\n"
#     '["variation 1", "variation 2"]\n\n'
#     'Complaint:\n"{text}"\n'
# )

# _GEMINI_SHORT_PROMPT = (
#     "Perform sentence-level rewriting on the following civic complaint." 
#     "No two sentence-level rewritings should be the same."
#     "Use synonymous words for some words in the sentence based on the context, while preserving the original meaning, "
#     "sentiment, and complaint intent.\n"
#     "Generate {n} different natural-language variations.\n\n"
#     "Return ONLY a valid JSON array of strings and nothing else.\n\n"
#     "Example output format:\n"
#     '["variation 1", "variation 2"]\n\n'
#     'Complaint:\n"{text}"\n'
# )


# # ─────────────────────────────────────────────────────────────────────────────
# # HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _deduplicate(texts: list[str], original: str) -> list[str]:
#     # Remove duplicates and exact matches to the original (case-insensitive).
#     seen   = {original.lower()}
#     unique = []
#     for t in texts:
#         key = t.lower()
#         if key not in seen:
#             seen.add(key)
#             unique.append(t)
#     return unique


# def _clean_df(X: list, y: list, severity_score: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
#     frame = pd.DataFrame({"x": X, "y": y, "severity_score": severity_score}).dropna()
#     frame["x"] = frame["x"].astype(str).str.strip()
#     frame = frame[(frame["x"] != "") & (frame["x"].str.lower() != "nan")]
#     frame = frame.drop_duplicates()
#     return frame["x"].values, frame["y"].values, frame["severity_score"].values


# def _introduce_spelling_errors(text: str) -> str:
#     words = text.split()
#     if not words:
#         return text
    
#     num_words = len(words)
#     # Based on the length of the sentence, decide to introduce 2 or 3 spelling errors
#     num_errors = 3 if num_words > 50 else 1
    
#     # Select words suitable for spelling error injection (length > 2)
#     eligible_indices = [idx for idx, w in enumerate(words) if len(w) > 2]
#     if not eligible_indices:
#         eligible_indices = [idx for idx, w in enumerate(words) if len(w) > 1]
#         if not eligible_indices:
#             return text
            
#     num_errors = min(num_errors, len(eligible_indices))
#     selected_word_indices = np.random.choice(eligible_indices, size=num_errors, replace=False)
    
#     for idx in selected_word_indices:
#         word = words[idx]
#         w_list = list(word)
#         error_type = np.random.choice(['swap', 'replace', 'delete', 'insert'])
#         char_idx = np.random.randint(0, len(word))
        
#         if error_type == 'swap' and len(word) > 1:
#             swap_with = char_idx + 1 if char_idx < len(word) - 1 else char_idx - 1
#             w_list[char_idx], w_list[swap_with] = w_list[swap_with], w_list[char_idx]
#         elif error_type == 'delete' and len(word) > 2:
#             w_list.pop(char_idx)
#         elif error_type == 'replace':
#             random_char = chr(np.random.randint(97, 123))  # a-z
#             w_list[char_idx] = random_char
#         elif error_type == 'insert':
#             random_char = chr(np.random.randint(97, 123))
#             w_list.insert(char_idx, random_char)
            
#         words[idx] = "".join(w_list)
        
#     return " ".join(words)


# # ─────────────────────────────────────────────────────────────────────────────
# # LLM BASED AUGMENTATION ROUTING (via Vertex AI)
# # ─────────────────────────────────────────────────────────────────────────────

# def gemini_augment(text: str, is_long: bool, n: int = 2) -> list[str]:
#     text = str(text).strip()
#     if not text:
#         return []
    
#     prompt_template = _GEMINI_LONG_PROMPT if is_long else _GEMINI_SHORT_PROMPT
#     prompt = prompt_template.format(n=n, text=text)
    
#     try:
#         response = _gemini.generate_content(
#             prompt,
#             generation_config={
#                 "temperature": 0.7,
#                 "top_p": 0.95,
#                 "response_mime_type": "application/json",
#             }
#         )
#         try:
#             raw = response.text.strip()
#         except ValueError:
#             raw = "".join(
#                 part.text for part in response.candidates[0].content.parts
#             ).strip()
        
#         # Strip markdown code fences if present
#         if raw.startswith("```"):
#             raw = raw.split("```")[1]
#             if raw.startswith("json"):
#                 raw = raw[4:]
#             raw = raw.strip()

#         parsed = json.loads(raw)
#         if not isinstance(parsed, list):
#             return []

#         cleaned = [
#             str(item).strip() for item in parsed
#             if item and str(item).strip().lower() not in ("", "nan", text.lower())
#         ]
#         return _deduplicate(cleaned, text)

#     except Exception as e:
#         logging.warning(f"gemini_augment failed: {e}")
#         return []


# def gemini_augment_with_retry(
#     text:        str,
#     is_long:     bool,
#     n:           int   = 3,
#     max_retries: int   = 3,
#     backoff:     float = 2.0,
# ) -> list[str]:
#     # gemini_augment with exponential back-off for rate-limit errors.
#     for attempt in range(1, max_retries + 1):
#         results = gemini_augment(text, is_long=is_long, n=n)
#         if results:
#             return results
#         wait = backoff ** attempt
#         logging.warning(
#             f"Gemini attempt {attempt}/{max_retries} failed. "
#             f"Retrying in {wait:.1f}s..."
#         )
#         time.sleep(wait)
#     return []


# # ─────────────────────────────────────────────────────────────────────────────
# # TIER FACTOR
# # ─────────────────────────────────────────────────────────────────────────────

# def _augment_factor(count: int) -> int:
#     if 1000 <= count <= 2000:
#         return 1
#     elif 500 <= count < 1000:
#         return 2
#     elif 100 <= count < 500:
#         return 3
#     elif 1 <= count < 100:
#         return 5
#     else:
#         return 0


# # ─────────────────────────────────────────────────────────────────────────────
# # ROW-LEVEL CHECKPOINT HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _ckpt_load(ckpt_path: Path) -> tuple:
#     done_indices = set()
#     X_syn, y_syn, sev_syn = [], [], []
#     if ckpt_path.exists():
#         with open(ckpt_path, "r", encoding="utf-8") as fh:
#             for line in fh:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 try:
#                     rec = json.loads(line)
#                     done_indices.add(int(rec["row_idx"]))
#                     X_syn.extend(rec["texts"])
#                     y_syn.extend(rec["labels"])
#                     sev_syn.extend(rec.get("severity_scores", rec.get("severities", [])))
#                 except (json.JSONDecodeError, KeyError):
#                     pass
#         print(
#             f"  [RESUME] Checkpoint loaded: {len(done_indices)} rows already done, "
#             f"{len(X_syn)} synthetic rows recovered."
#         )
#     return done_indices, X_syn, y_syn, sev_syn


# def _ckpt_append(ckpt_path: Path, row_idx: int,
#                  texts: list, labels: list, severity_scores: list) -> None:
#     rec = {
#         "row_idx":         row_idx,
#         "texts":           texts,
#         "labels":          labels,
#         "severity_scores": severity_scores,
#     }
#     with open(ckpt_path, "a", encoding="utf-8") as fh:
#         fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


# # ─────────────────────────────────────────────────────────────────────────────
# # MAIN AUGMENTATION ENTRY POINT
# # ─────────────────────────────────────────────────────────────────────────────

# def augment_dataset(
#     X:                     np.ndarray,
#     y:                     np.ndarray,
#     severity_score:        np.ndarray,
#     checkpoint_path:       Path | None = None,
#     class_counts_override: dict | None = None,
#     preprocess_fn:         callable = None,
# ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    
#     # ── Load checkpoint (if any) ──────────────────────────────────────────────
#     if checkpoint_path is not None:
#         checkpoint_path = Path(checkpoint_path)
#         checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
#         done_indices, X_syn, y_syn, sev_syn = _ckpt_load(checkpoint_path)
#     else:
#         done_indices, X_syn, y_syn, sev_syn = set(), [], [], []

#     unique_classes, counts = np.unique(y, return_counts=True)

#     for cls, count in zip(unique_classes, counts):
#         factor_count = class_counts_override.get(cls, count) if class_counts_override else count
#         factor       = _augment_factor(int(factor_count))
#         cls_mask     = (y == cls)
#         cls_indices  = np.where(cls_mask)[0]

#         pending = [int(i) for i in cls_indices if int(i) not in done_indices]

#         cls_lengths = np.array([len(str(t).split()) for t in X[cls_mask]])
#         long_mask   = cls_lengths > 110
#         short_mask = ~long_mask

#         print(
#             f"\n  Class '{cls}' | {count:,} samples | "
#             f"long>110: {long_mask.sum()} | short<=110: {short_mask.sum()} | "
#             f"factor: x{factor} | pending: {len(pending)}"
#         )

#         if factor == 0:
#             print(f"  [SKIP] Class '{cls}' has {count:,} samples (>2000) -- no augmentation needed.")
#             continue

#         if not pending:
#             print(f"  [OK] Class '{cls}' fully restored from checkpoint -- skipping.")
#             continue

#         with tqdm(
#             total      = len(pending),
#             desc       = f"  Class '{cls}'",
#             unit       = "row",
#             colour     = "cyan",
#             leave      = True,
#             bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} rows "
#                          "[{elapsed}<{remaining}, {rate_fmt}]  "
#                          "syn_rows={postfix[0]}",
#             postfix    = [len(X_syn)],
#         ) as pbar:

#             for i in pending:
#                 complaint_len = len(str(X[i]).split())
#                 is_long = complaint_len > 110

#                 row_texts:  list = []
#                 row_labels: list = []
#                 row_sevs:   list = []

#                 # Request factor * 2 options in 1 API call to handle filtering / duplicates efficiently
#                 new_texts = gemini_augment_with_retry(X[i], is_long=is_long, n=factor * 2)
#                 new_texts = new_texts[:factor]

#                 for t in new_texts:
#                     if preprocess_fn is not None:
#                         t = preprocess_fn(t)
#                     if t and t.strip():
#                         row_texts.append(t)
#                         row_labels.append(str(cls))
#                         row_sevs.append(str(severity_score[i]))

#                 pbar.update(1)

#                 # ── Accumulate in-memory lists ────────────────────────────────
#                 X_syn.extend(row_texts)
#                 y_syn.extend(row_labels)
#                 sev_syn.extend(row_sevs)
#                 pbar.postfix[0] = len(X_syn)

#                 # ── Write checkpoint line AFTER the row is fully done ─────────
#                 if checkpoint_path is not None:
#                     _ckpt_append(
#                         checkpoint_path,
#                         row_idx         = i,
#                         texts           = row_texts,
#                         labels          = row_labels,
#                         severity_scores = row_sevs,
#                     )

#                 # Polite rate-limiting delay between requests
#                 time.sleep(1.0)

#     # ── Step 3: Randomly select ~5% of synthetic complaints to introduce spelling errors ──
#     # Done only on X_syn to guarantee that the original rows are kept exactly preserved.
#     if len(X_syn) > 0:
#         num_to_select = int(round(0.05 * len(X_syn)))
#         if num_to_select > 0:
#             # Weighted probability: longer sentences (by word count) have a higher chance of selection
#             syn_lengths = np.array([len(x.split()) for x in X_syn])
#             total_len = syn_lengths.sum()
#             if total_len > 0:
#                 probs = syn_lengths / total_len
#             else:
#                 probs = np.ones(len(X_syn)) / len(X_syn)
            
#             selected_indices = np.random.choice(len(X_syn), size=num_to_select, replace=False, p=probs)
#             for idx in selected_indices:
#                 X_syn[idx] = _introduce_spelling_errors(X_syn[idx])

#     # ── Merge originals (always kept) + synthetics ────────────────────────────
#     X_merged   = list(X)             + X_syn
#     y_merged   = list(y)             + y_syn
#     sev_merged = list(severity_score) + sev_syn

#     # ── Final dedup / clean pass ──────────────────────────────────────────────
#     X_out, y_out, severity_out = _clean_df(X_merged, y_merged, sev_merged)

#     assert len(X_out) == len(y_out) == len(severity_out), \
#         "Length mismatch after augmentation"
#     assert all(isinstance(x, str) and x.strip() for x in X_out), \
#         "Empty or non-string texts found after augmentation"

#     n_orig  = len(X)
#     n_added = len(X_out) - n_orig
#     print(f"\n  Original : {n_orig:,}")
#     print(f"  Synthetic: {n_added:,}")
#     print(f"  Total    : {len(X_out):,}")

#     return X_out, y_out, severity_out


In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # 5-FOLD STRATIFIED CROSS-VALIDATION -- Civic Agency Classification
# # ─────────────────────────────────────────────────────────────────────────────
# #
# # Workflow:
# #
# # 1. Create two preprocessed datasets from the original dataset:
# #    - Classical ML Dataset     → preprocess_classical_ml()
# #    - BERT NLP Dataset         → preprocess_nlp_bert()
# #
# # 2. Split each dataset into 5 stratified folds.
# #
# # 3. Augment each fold independently and only once:
# #    - Classical ML folds → df_classical_aug_1 ... df_classical_aug_5
# #    - BERT NLP folds     → df_bert_aug_1 ... df_bert_aug_5
# #
# # 4. For each CV iteration:
# #    - Training   = 4 augmented folds
# #    - Validation = 1 original (non-augmented) fold
# #
# # 5. Repeat Step 4 for all 5 folds.
# #
# # 6. Save the fold-wise train/validation splits as:
# #    - df_classical_ml_cv.joblib
# #    - df_bert_nlp_cv.joblib
# #
# # Result:
# #    - Classical ML CV Dataset
# #      (preprocess_classical_ml + augmentation)
# #
# #    - BERT NLP CV Dataset
# #      (preprocess_nlp_bert + augmentation)
# # ─────────────────────────────────────────────────────────────────────────────

# # Exact save paths from Step 6
# SAVE_PATH_CLASSICAL = PROJECT_ROOT / "data" / "processed" / "df_classical_ml_cv.joblib"
# SAVE_PATH_BERT      = PROJECT_ROOT / "data" / "processed" / "df_bert_nlp_cv.joblib"

# SAVE_PATH_CLASSICAL.parent.mkdir(parents=True, exist_ok=True)
# SAVE_PATH_BERT.parent.mkdir(parents=True, exist_ok=True)

# CKPT_DIR  = PROJECT_ROOT / "data" / "processed" / "aug_checkpoints_v2"
# PARTS_DIR = PROJECT_ROOT / "data" / "processed" / "aug_parts_df_v2"
# CKPT_DIR.mkdir(parents=True, exist_ok=True)
# PARTS_DIR.mkdir(parents=True, exist_ok=True)

# if SAVE_PATH_CLASSICAL.exists() and SAVE_PATH_BERT.exists():
#     print(f"[OK] Loading cached CV splits:")
#     fold_data_list_classical = joblib.load(SAVE_PATH_CLASSICAL)
#     fold_data_list_bert      = joblib.load(SAVE_PATH_BERT)
#     print(f"   Loaded Classical ML CV Dataset: {len(fold_data_list_classical)} folds.")
#     print(f"   Loaded BERT NLP CV Dataset: {len(fold_data_list_bert)} folds.")

# else:
#     print("[INFO] Running 5-Fold Stratified CV Pipeline...\n")

#     # Assuming dataframe `df` is available. 
#     # NOTE: Using "description" and "civic_agency_title" if your column names differ!
#     y_arr = df["civic_agency_title"].values
#     severity_arr = df["severity_score"].values

#     _cls_unique, _cls_counts = np.unique(y_arr, return_counts=True)
#     full_class_counts = dict(zip(_cls_unique, _cls_counts.astype(int)))

#     # =========================================================================
#     # Step 1: Create two preprocessed datasets from the original dataset
#     # =========================================================================
#     print("Step 1: Creating preprocessed datasets...")
#     X_classical = np.array([preprocess_classical_ml(str(t)) for t in df["description"].values])
#     X_bert      = np.array([preprocess_nlp_bert(str(t))     for t in df["description"].values])

#     # =========================================================================
#     # Step 2: Split each dataset into 5 stratified folds
#     # =========================================================================
#     print("Step 2: Splitting into 5 stratified folds...")
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
#     # The split indices are identical for both datasets since they share `y_arr`
#     all_splits = list(skf.split(np.zeros(len(y_arr)), y_arr))

#     # =========================================================================
#     # Step 3: Augment each fold independently and only once
#     # =========================================================================
#     print("Step 3: Augmenting each fold independently...")
    
#     df_classical_aug_parts = []
#     df_bert_aug_parts      = []
#     original_parts         = []

#     for part_idx in range(5):
#         _, part_indices = all_splits[part_idx]

#         # Extract fold slices
#         part_y    = y_arr[part_indices]
#         part_sev  = severity_arr[part_indices]
        
#         part_X_classical = X_classical[part_indices]
#         part_X_bert      = X_bert[part_indices]

#         # Save the original un-augmented slices for the validation sets later
#         original_parts.append({
#             "X_classical":    part_X_classical,
#             "X_bert":         part_X_bert,
#             "y":              part_y,
#             "severity_score": part_sev,
#             "idx":            part_indices
#         })

#         # --- Augment Classical ML Fold ---
#         part_save_classical = PARTS_DIR / f"df_classical_aug_{part_idx+1}.joblib"
#         if part_save_classical.exists():
#             aug_classical = joblib.load(part_save_classical)
#         else:
#             print(f"\n--- Augmenting Classical ML Fold {part_idx+1} ---")
#             part_ckpt_classical = CKPT_DIR / f"classical_part_{part_idx+1}.jsonl"
#             aug_X_class, aug_y_class, aug_sev_class = augment_dataset(
#                 X                     = part_X_classical,
#                 y                     = part_y,
#                 severity_score        = part_sev,
#                 checkpoint_path       = part_ckpt_classical,
#                 preprocess_fn         = preprocess_classical_ml,
#                 class_counts_override = full_class_counts,
#             )
#             aug_classical = {"X": aug_X_class, "y": aug_y_class, "severity_score": aug_sev_class}
#             joblib.dump(aug_classical, part_save_classical, compress=3)
#             if part_ckpt_classical.exists(): part_ckpt_classical.unlink()
        
#         df_classical_aug_parts.append(aug_classical)

#         # --- Augment BERT NLP Fold ---
#         part_save_bert = PARTS_DIR / f"df_bert_aug_{part_idx+1}.joblib"
#         if part_save_bert.exists():
#             aug_bert = joblib.load(part_save_bert)
#         else:
#             print(f"\n--- Augmenting BERT NLP Fold {part_idx+1} ---")
#             part_ckpt_bert = CKPT_DIR / f"bert_part_{part_idx+1}.jsonl"
#             aug_X_bert, aug_y_bert, aug_sev_bert = augment_dataset(
#                 X                     = part_X_bert,
#                 y                     = part_y,
#                 severity_score        = part_sev,
#                 checkpoint_path       = part_ckpt_bert,
#                 preprocess_fn         = preprocess_nlp_bert,
#                 class_counts_override = full_class_counts,
#             )
#             aug_bert = {"X": aug_X_bert, "y": aug_y_bert, "severity_score": aug_sev_bert}
#             joblib.dump(aug_bert, part_save_bert, compress=3)
#             if part_ckpt_bert.exists(): part_ckpt_bert.unlink()
            
#         df_bert_aug_parts.append(aug_bert)

#     # =========================================================================
#     # Step 4 & 5: For each CV iteration, assemble Training (4) & Validation (1)
#     # =========================================================================
#     print("\nSteps 4 & 5: Assembling CV iterations...")

#     df_classical_ml_cv = []
#     df_bert_nlp_cv     = []

#     for fold_idx in range(5):
#         # Build Classical Training Set (4 augmented folds)
#         train_X_class_parts   = [df_classical_aug_parts[i]["X"]              for i in range(5) if i != fold_idx]
#         train_y_class_parts   = [df_classical_aug_parts[i]["y"]              for i in range(5) if i != fold_idx]
#         train_sev_class_parts = [df_classical_aug_parts[i]["severity_score"] for i in range(5) if i != fold_idx]

#         # Build BERT Training Set (4 augmented folds)
#         train_X_bert_parts   = [df_bert_aug_parts[i]["X"]              for i in range(5) if i != fold_idx]
#         train_y_bert_parts   = [df_bert_aug_parts[i]["y"]              for i in range(5) if i != fold_idx]
#         train_sev_bert_parts = [df_bert_aug_parts[i]["severity_score"] for i in range(5) if i != fold_idx]

#         # Assemble Classical Dict
#         df_classical_ml_cv.append({
#             "train_X":              np.concatenate(train_X_class_parts),
#             "train_y":              np.concatenate(train_y_class_parts),
#             "train_severity_score": np.concatenate(train_sev_class_parts),
#             "val_X":                original_parts[fold_idx]["X_classical"],
#             "val_y":                original_parts[fold_idx]["y"],
#             "val_severity_score":   original_parts[fold_idx]["severity_score"],
#             "val_idx":              original_parts[fold_idx]["idx"],
#         })

#         # Assemble BERT Dict
#         df_bert_nlp_cv.append({
#             "train_X":              np.concatenate(train_X_bert_parts),
#             "train_y":              np.concatenate(train_y_bert_parts),
#             "train_severity_score": np.concatenate(train_sev_bert_parts),
#             "val_X":                original_parts[fold_idx]["X_bert"],
#             "val_y":                original_parts[fold_idx]["y"],
#             "val_severity_score":   original_parts[fold_idx]["severity_score"],
#             "val_idx":              original_parts[fold_idx]["idx"],
#         })

#         print(f"  CV Iteration {fold_idx+1}/5 assembled.")

#     # =========================================================================
#     # Step 6: Save the fold-wise train/validation splits
#     # =========================================================================
#     joblib.dump(df_classical_ml_cv, SAVE_PATH_CLASSICAL, compress=3)
#     print(f"\nStep 6 [OK] Saved -> {SAVE_PATH_CLASSICAL.name}")

#     joblib.dump(df_bert_nlp_cv, SAVE_PATH_BERT, compress=3)
#     print(f"Step 6 [OK] Saved -> {SAVE_PATH_BERT.name}")

### 3.6  Analysis of the synthesised datsets.

In [ ]:
# Paths supporting both root-level and notebook-level execution
paths_config = {
    "bert_nlp": {
        "data_path": [
            Path("data/processed/df_bert_nlp_cv.joblib"),
            Path("../data/processed/df_bert_nlp_cv.joblib")
        ],
        "image_name": "3.6a_data_profile_bert_nlp.png"
    },
    "classical_ml": {
        "data_path": [
            Path("data/processed/df_classical_ml_cv.joblib"),
            Path("../data/processed/df_classical_ml_cv.joblib")
        ],
        "image_name": "3.6b_data_profile_classical_ml.png"
    }
}

# Ensure output directory exists (handles root and notebook paths)
charts_dir = Path("charts_and_graphs/civic_agency_results")
if not charts_dir.exists():
    charts_dir = Path("../charts_and_graphs/civic_agency_results")
charts_dir.mkdir(parents=True, exist_ok=True)

def generate_profile_chart(model_key, config):
    # Find the data file
    resolved_path = None
    for p in config["data_path"]:
        if p.exists():
            resolved_path = p
            break
            
    if not resolved_path:
        print(f"❌ Data file not found for {model_key}")
        return
        
    print(f"Generating chart for {model_key} from {resolved_path}...")
    
    # Load dataset
    data = joblib.load(resolved_path)
    
    # Extract Fold 0 splits
    fold_0 = data[0]
    train_y = fold_0["train_y"]
    val_y = fold_0["val_y"]
    
    # Compute counts
    train_counts = pd.Series(train_y).value_counts().reset_index()
    train_counts.columns = ["Civic Agency", "Count"]
    train_counts["Split"] = "Train (Augmented)"
    
    val_counts = pd.Series(val_y).value_counts().reset_index()
    val_counts.columns = ["Civic Agency", "Count"]
    val_counts["Split"] = "Validation (Original)"
    
    # Combine into a single DataFrame for Seaborn
    plot_df = pd.concat([train_counts, val_counts], ignore_index=True)
    
    # Plotting setup
    plt.figure(figsize=(12, 6))
    sns.set_theme(style="whitegrid")
    
    # Premium color palette: deep blue for train, teal for validation
    colors = ["#3f51b5", "#009688"]
    
    ax = sns.barplot(
        x="Count", 
        y="Civic Agency", 
        hue="Split", 
        data=plot_df, 
        palette=colors,
        edgecolor="white",
        linewidth=1
    )
    
    # Customizing aesthetics
    title_suffix = "BERT/NLP Pipeline" if model_key == "bert_nlp" else "Classical ML Pipeline"
    plt.title(f"Class Balance Profile - {title_suffix} (Fold 0)", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Number of Samples", fontsize=12)
    plt.ylabel("Civic Agency", fontsize=12)
    
    # Add values on top of the bars
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3, fontsize=9, fontweight="semibold")
        
    plt.legend(title="Split", loc="lower right", frameon=True, shadow=False)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    
    # Save the output image
    save_path = charts_dir / config["image_name"]
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"✅ Chart saved successfully at: {save_path}\n")

# Run chart generator
for key, cfg in paths_config.items():
    generate_profile_chart(key, cfg)


### 3.7  Rebuilding the final datasets

In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # Clean Augmented Data (with severity_reason + complaint_length)
# # ─────────────────────────────────────────────────────────────────────────────

# CLEAN_MODES = [
#     {
#         "aug_path": PROJECT_ROOT / "data" / "processed" / "df_classical_ml_cv.joblib",
#         "final_path": PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final.joblib",
#         "name": "Classical ML"
#     },
#     {
#         "aug_path": PROJECT_ROOT / "data" / "processed" / "df_bert_nlp_cv.joblib",
#         "final_path": PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final_bert.joblib",
#         "name": "BERT NLP"
#     }
# ]

# def clean_fold_train(X, y, severity, reason):
#     """Remove NaN, blank, duplicate, and < 3 word samples from training arrays."""
#     df_temp = pd.DataFrame({"X": X, "y": y, "severity": severity, "reason": reason})
#     before = len(df_temp)

#     # 1. Remove NaN / blank
#     mask_valid = df_temp["X"].apply(
#         lambda x: isinstance(x, str) and x.strip() != "" and x.strip().lower() != "nan"
#     )
#     df_temp = df_temp[mask_valid]
#     dropped_nan = before - len(df_temp)

#     # 2. Remove duplicates (keep first occurrence)
#     before_dup = len(df_temp)
#     df_temp = df_temp.drop_duplicates(subset=["X"], keep="first")
#     dropped_dup = before_dup - len(df_temp)

#     # 3. Remove < 3 word complaints
#     before_short = len(df_temp)
#     df_temp = df_temp[df_temp["X"].apply(lambda x: len(x.split()) >= 3)]
#     dropped_short = before_short - len(df_temp)

#     # 4. Compute complaint_length (word count)
#     complaint_length = df_temp["X"].apply(lambda x: len(x.split())).values

#     print(f"    Removed -> NaN/blank: {dropped_nan}, duplicates: {dropped_dup}, "
#           f"< 3 words: {dropped_short}  |  {before:,} -> {len(df_temp):,}")

#     return (df_temp["X"].values, df_temp["y"].values,
#             df_temp["severity"].values, df_temp["reason"].values,
#             complaint_length)

# for cfg in CLEAN_MODES:
#     print(f"\n============================================================")
#     print(f"  CLEANING AUGMENTED DATA FOR: {cfg['name']}")
#     print(f"============================================================")
    
#     if not cfg["aug_path"].exists():
#         print(f"[WARNING] Augmented path does not exist, skipping: {cfg['aug_path']}")
#         continue
        
#     _aug_data = joblib.load(cfg["aug_path"])
#     fold_data_clean = []

#     for fi, fold in enumerate(_aug_data):
#         print(f"  Fold {fi}:")
#         clean_X, clean_y, clean_sev, clean_reason, clean_len = clean_fold_train(
#             fold["train_X"], fold["train_y"],
#             fold["train_severity_score"], fold["train_severity_reason"]
#         )

#         # Compute complaint_length for validation set too
#         val_len = np.array([len(str(x).split()) for x in fold["val_X"]])

#         fold_data_clean.append({
#             "train_X":                clean_X,
#             "train_y":                clean_y,
#             "train_severity_score":   clean_sev,
#             "train_severity_reason":  clean_reason,
#             "train_complaint_length": clean_len,
#             "val_X":                  fold["val_X"],
#             "val_y":                  fold["val_y"],
#             "val_severity_score":     fold["val_severity_score"],
#             "val_severity_reason":    fold["val_severity_reason"],
#             "val_complaint_length":   val_len,
#             "val_idx":                fold["val_idx"],
#         })

#     joblib.dump(fold_data_clean, cfg["final_path"], compress=3)
#     print(f"\n  [OK] Saved cleaned folds -> {cfg['final_path'].name}")
#     print(f"       Keys: {list(fold_data_clean[0].keys())}")

### 3.7  Shared Utilities for Hyperparameter Optimsation using Bayesian Opmtimisation Optuna:

In [ ]:
# Suppress Optuna verbose logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 43
random.seed(SEED)
np.random.seed(SEED)

# ── Load final dataset relative to notebook/ folder ───────────────────────────
fold_data_path = Path("../data/processed/df_final_claasical_ml_v2.joblib")
fold_data = joblib.load(fold_data_path)
print(f"✅ Civic agency final fold data loaded from: {fold_data_path}")

# Set up global target indices and classes
total_size = sum(len(d["val_y"]) for d in fold_data)
y_global = np.empty(total_size, dtype=object)
for d in fold_data:
    y_global[d["val_idx"]] = d["val_y"]
labels = sorted(np.unique(y_global))
print(f"Classes ({len(labels)}): {labels}")

# Helper for file versioning
def _next_version(path: Path, base_name: str) -> int:
    i = 1
    while (path / f"{base_name}_{i}.json").exists():
        i += 1
    return i

# ── Optuna Study-Level Early Stopping Callback ────────────────────────────────
class StudyEarlyStoppingCallback:
    """Callback to stop the study if validation F1 has not improved for `patience` trials."""
    def __init__(self, patience=15):
        self.patience = patience
        self.best_value = None
        self.trials_without_improvement = 0

    def __call__(self, study, trial):
        if study.best_value is None:
            return
            
        if self.best_value is None or study.best_value > self.best_value:
            self.best_value = study.best_value
            self.trials_without_improvement = 0
        else:
            self.trials_without_improvement += 1
            if self.trials_without_improvement >= self.patience:
                print(f"🛑 Study-level early stopping triggered: No improvement for {self.patience} trials.")
                study.stop()

# ── Reusable Optuna Optimization Search Function ──────────────────────────────
def run_optuna_search(
    model_name, 
    build_fn, 
    define_trial_params_fn, 
    fold_data, 
    y_global, 
    labels, 
    output_root, 
    n_trials=100,
    patience=15
):
    """
    Cross-validated Bayesian Optimization using Optuna.
    Saves best params, OOF confusion matrix PNG, and study metrics JSON.
    """
    out_dir = output_root / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    version = _next_version(out_dir, f"optuna_results_{model_name}")
    
    print(f"\n{'='*65}\n  OPTUNA BAYESIAN OPTIMIZATION — {model_name.upper()}\n{'='*65}")

    # 1. Define objective function with fold-level trial pruning
    def objective(trial):
        params = define_trial_params_fn(trial)
        val_f1_macros = []
        
        for fold_idx, d in enumerate(fold_data):
            random.seed(SEED)
            np.random.seed(SEED)
            
            model = build_fn()
            model.set_params(**params)
            model.fit(d["train_X"], d["train_y"])
            
            val_preds = model.predict(d["val_X"])
            fold_f1 = f1_score(d["val_y"], val_preds, average="macro")
            val_f1_macros.append(fold_f1)
            
            # Report fold score to the pruner
            trial.report(fold_f1, step=fold_idx)
            
            # Check for early pruning of this trial (Raise optuna.TrialPruned)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
        return float(np.mean(val_f1_macros))

    # 2. Run the study
    study = optuna.create_study(
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    )
    
    early_stop_cb = StudyEarlyStoppingCallback(patience=patience)
    study.optimize(objective, n_trials=n_trials, callbacks=[early_stop_cb])
    
    # Extract results
    complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not complete_trials:
        print("⚠️ All trials were pruned! Using baseline parameters.")
        best_params = study.trials[0].params
        best_f1 = 0.0
    else:
        best_params = study.best_params
        best_f1 = study.best_value
        
    print(f"\n🏆 Best Val F1-macro: {best_f1:.4f}")
    print(f"   Best Parameters: {best_params}")

    # 3. Evaluate best model on all folds to extract metrics
    print("\n--- Profiling Best Parameters ---")
    train_accs, train_precs, train_recs, train_f1s = [], [], [], []
    val_accs, val_precs, val_recs, val_f1s = [], [], [], []
    
    for fi, d in enumerate(fold_data):
        random.seed(SEED)
        np.random.seed(SEED)
        
        model = build_fn()
        model.set_params(**best_params)
        model.fit(d["train_X"], d["train_y"])
        
        train_preds = model.predict(d["train_X"])
        train_accs.append(accuracy_score(d["train_y"], train_preds))
        train_precs.append(precision_score(d["train_y"], train_preds, average="macro", zero_division=0))
        train_recs.append(recall_score(d["train_y"], train_preds, average="macro", zero_division=0))
        train_f1s.append(f1_score(d["train_y"], train_preds, average="macro"))
        
        val_preds = model.predict(d["val_X"])
        val_accs.append(accuracy_score(d["val_y"], val_preds))
        val_precs.append(precision_score(d["val_y"], val_preds, average="macro", zero_division=0))
        val_recs.append(recall_score(d["val_y"], val_preds, average="macro", zero_division=0))
        val_f1s.append(f1_score(d["val_y"], val_preds, average="macro"))
        
    best_metrics = {
        "val_f1_macro": float(np.mean(val_f1s)),
        "val_accuracy": float(np.mean(val_accs)),
        "val_precision": float(np.mean(val_precs)),
        "val_recall": float(np.mean(val_recs)),
        "train_f1_macro": float(np.mean(train_f1s)),
        "train_accuracy": float(np.mean(train_accs)),
        "train_precision": float(np.mean(train_precs)),
        "train_recall": float(np.mean(train_recs)),
    }
    
    print(f"    Val   | F1-macro: {best_metrics['val_f1_macro']:.4f} | Acc: {best_metrics['val_accuracy']:.4f} | P: {best_metrics['val_precision']:.4f} | R: {best_metrics['val_recall']:.4f}")
    print(f"    Train | F1-macro: {best_metrics['train_f1_macro']:.4f} | Acc: {best_metrics['train_accuracy']:.4f} | P: {best_metrics['train_precision']:.4f} | R: {best_metrics['train_recall']:.4f}")

    # 4. Generate OOF predictions & confusion matrix
    oof = np.empty_like(y_global, dtype=object)
    for d in fold_data:
        model = build_fn()
        model.set_params(**best_params)
        model.fit(d["train_X"], d["train_y"])
        oof[d["val_idx"]] = model.predict(d["val_X"])
        
    cm = confusion_matrix(y_global, oof, labels=labels)
    
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=False)
    plt.title(f"Confusion Matrix — {model_name} (Optuna v{version})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    cm_path = out_dir / f"3.8_optuna_confusion_matrix_{model_name}_{version}.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
    # 5. Export results to JSON
    results = {
        "version": version, 
        "model": model_name,
        "best_params": best_params,
        "metrics": best_metrics,
        "n_trials": len(study.trials)
    }
    
    json_path = out_dir / f"optuna_results_{model_name}_{version}.json"
    with open(json_path, "w") as fh:
        json.dump(results, fh, indent=4)
        
    print(f"💾 Saved Optuna metrics -> {json_path.name}")
    print(f"🖼️ Saved Confusion Matrix -> {cm_path.name}")
    
    return best_params, best_metrics

### 3.8  Logistic Regression — Civic Agency


In [ ]:
# 1. Define model builder using standard LogisticRegression
def build_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), stop_words="english")),
        ("lr", LogisticRegression(max_iter=1000, random_state=SEED))
    ])

# 2. Define hyperparameter search space (tuning C, class weights, and TF-IDF parameters)
def lr_search_space(trial):
    return {
        "tfidf__max_features": trial.suggest_int("tfidf__max_features", 1000, 12000, step=1000),
        "tfidf__min_df": trial.suggest_int("tfidf__min_df", 1, 10),
        "tfidf__max_df": trial.suggest_float("tfidf__max_df", 0.7, 1.0, step=0.05),
        "lr__C": trial.suggest_float("lr__C", 0.01, 100.0, log=True),
        "lr__class_weight": trial.suggest_categorical("lr__class_weight", [None, "balanced"])
    }

# 3. Execute Bayesian search
output_dir = Path("models/civic_bodies/dataset_v2")
best_params, best_metrics = run_optuna_search(
    model_name="logistic_regression",
    build_fn=build_lr,
    define_trial_params_fn=lr_search_space,
    fold_data=fold_data,
    y_global=y_global,
    labels=labels,
    output_root=output_dir,
    n_trials=100,
    patience=15  # Stop entire search early if no study improvement for 15 trials
)


#### Complaint-Length vs Misclassification

We test statistically whether shorter complaints are harder to classify correctly. This guides future feature engineering (e.g. minimum-length filters, length-aware models).


In [ ]:
all_texts, all_true, all_pred = [], [], []

# Use the best parameters found by Optuna
for d in fold_data:
    # Build pipeline matching the same step names as build_lr()
    m = Pipeline([
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("lr",    LogisticRegression(solver="lbfgs", max_iter=2000, tol=1e-3, n_jobs=-1, random_state=SEED)),
    ])
    
    # Apply the best parameters from Optuna search
    m.set_params(**best_params)
    m.fit(d["train_X"], d["train_y"])
    
    preds = m.predict(d["val_X"])
    all_texts.extend(d["val_X"])
    all_true.extend(d["val_y"])
    all_pred.extend(preds)

# Analyze complaint length vs. misclassifications
analysis_df = pd.DataFrame({
    "text": all_texts, 
    "true_label": all_true, 
    "pred_label": all_pred,
    "complaint_length": [len(str(t).split()) for t in all_texts],
})
analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# Statistical significance tests
_, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
_, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
n1, n2 = len(correct_len), len(miscls_len)
pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

size = ("Negligible" if abs(cohens_d) < 0.2 else
        "Small"      if abs(cohens_d) < 0.5 else
        "Medium"     if abs(cohens_d) < 0.8 else "Large")

print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
      f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")

### 3.8  LinearSVC — Civic Agency


In [ ]:
# 1. Define model builder function
def build_svc():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), stop_words="english")),
        ("svc",   LinearSVC(penalty="l2", multi_class="ovr", max_iter=2000, dual="auto", random_state=SEED))
    ])

# 2. Define hyperparameter search space (tuning C, class weights, and pruning parameters)
def svc_search_space(trial):
    return {
        "tfidf__max_features": trial.suggest_int("tfidf__max_features", 1000, 12000, step=1000),
        "tfidf__min_df": trial.suggest_int("tfidf__min_df", 1, 10),
        "tfidf__max_df": trial.suggest_float("tfidf__max_df", 0.7, 1.0, step=0.05),
        "svc__C": trial.suggest_float("svc__C", 0.01, 10.0, log=True),
        "svc__class_weight": trial.suggest_categorical("svc__class_weight", [None, "balanced"])
    }

# 3. Execute Bayesian search
best_params_svc, best_metrics_svc = run_optuna_search(
    model_name="linearsvc",
    build_fn=build_svc,
    define_trial_params_fn=svc_search_space,
    fold_data=fold_data,
    y_global=y_global,
    labels=labels,
    output_root=output_dir,
    n_trials=100,
    patience=15
)


In [ ]:
all_texts, all_true, all_pred = [], [], []

# Use the best parameters found by Optuna for LinearSVC
for d in fold_data:
    # Build pipeline matching the same step names as build_svc()
    m = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3))),
        ("svc",   LinearSVC(penalty="l2", multi_class="ovr", max_iter=2000, dual="auto", random_state=SEED)),
    ])
    
    # Apply the best parameters from Optuna search
    m.set_params(**best_params_svc)
    m.fit(d["train_X"], d["train_y"])
    
    preds = m.predict(d["val_X"])
    all_texts.extend(d["val_X"])
    all_true.extend(d["val_y"])
    all_pred.extend(preds)

# Analyze complaint length vs. misclassifications
analysis_df = pd.DataFrame({
    "text": all_texts, 
    "true_label": all_true, 
    "pred_label": all_pred,
    "complaint_length": [len(str(t).split()) for t in all_texts],
})
analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# Statistical significance tests
_, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
_, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
n1, n2 = len(correct_len), len(miscls_len)
pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

size = ("Negligible" if abs(cohens_d) < 0.2 else
        "Small"      if abs(cohens_d) < 0.5 else
        "Medium"     if abs(cohens_d) < 0.8 else "Large")

print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
      f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")


### 3.9  MultinomialNB — Civic Agency


In [ ]:
# 1. Define model builder function
def build_nb():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), stop_words="english")),
        ("nb",    MultinomialNB())
    ])

# 2. Define hyperparameter search space (tuning alpha, priors, and pruning parameters)
def nb_search_space(trial):
    return {
        "tfidf__max_features": trial.suggest_int("tfidf__max_features", 1000, 12000, step=1000),
        "tfidf__min_df": trial.suggest_int("tfidf__min_df", 1, 10),
        "tfidf__max_df": trial.suggest_float("tfidf__max_df", 0.7, 1.0, step=0.05),
        "nb__alpha": trial.suggest_float("nb__alpha", 1e-3, 10.0, log=True),  # Laplace smoothing
        "nb__fit_prior": trial.suggest_categorical("nb__fit_prior", [True, False])
    }

# 3. Execute Bayesian search
best_params_nb, best_metrics_nb = run_optuna_search(
    model_name="multinomial_nb",
    build_fn=build_nb,
    define_trial_params_fn=nb_search_space,
    fold_data=fold_data,
    y_global=y_global,
    labels=labels,
    output_root=output_dir,
    n_trials=100,
    patience=15
)


In [ ]:
all_texts, all_true, all_pred = [], [], []

# Use the best parameters found by Optuna for MultinomialNB
for d in fold_data:
    # Build pipeline matching the same step names as build_nb()
    m = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3))),
        ("nb",    MultinomialNB()),
    ])
    
    # Apply the best parameters from Optuna search
    m.set_params(**best_params_nb)
    m.fit(d["train_X"], d["train_y"])
    
    preds = m.predict(d["val_X"])
    all_texts.extend(d["val_X"])
    all_true.extend(d["val_y"])
    all_pred.extend(preds)

# Analyze complaint length vs. misclassifications
analysis_df = pd.DataFrame({
    "text": all_texts, 
    "true_label": all_true, 
    "pred_label": all_pred,
    "complaint_length": [len(str(t).split()) for t in all_texts],
})
analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# Statistical significance tests
_, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
_, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
n1, n2 = len(correct_len), len(miscls_len)
pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

size = ("Negligible" if abs(cohens_d) < 0.2 else
        "Small"      if abs(cohens_d) < 0.5 else
        "Medium"     if abs(cohens_d) < 0.8 else "Large")

print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
      f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")


### 3.10  DistilBERT Model — Civic Agency (Dataset V2)

We fine-tune the pre-trained `distilbert-base-uncased` model using 5-fold cross-validation. This model is lightweight and serves as a fast baseline deep learning model.

> **[!TIP]**
> * **Training Script:** [`scripts/distilbert_train_civic_v2.py`](../scripts/distilbert_train_civic_v2.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_civic_v2.py`](../scripts/submit_vertex_job_civic_v2.py)


In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../models/civic_bodies/dataset_v2/DistilBERT/oof_predictions_civic.joblib")

if oof_path.exists():
    data = joblib.load(oof_path)
    y_true = data["true"]
    y_pred = data["pred"]
    labels = data["labels"]

    # 1. Compute per-class binary accuracy and standard classification metrics
    class_data = []
    for label in sorted(labels):
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None
        )
        support = s_val[0]
        
        class_data.append({
            "Civic Agency": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Civic Agency")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # 2. Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("DistilBERT - Per-Agency Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Civic Agency & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DistilBERT - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    # Save to model output folder and render
    save_path = oof_path.parent / "3.10_distilbert_per_agency_metrics.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_civic.joblib not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/civic_bodies/dataset_v2/DistilBERT/results_distilbert_civic.json")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Aggregate standard metrics per fold
        fold_data.append({"Fold": fold_label, "Metric": "Accuracy", "Value": item["val_accuracy"]})
        fold_data.append({"Fold": fold_label, "Metric": "Precision", "Value": item.get("val_precision") or item.get("val_precision_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "Recall", "Value": item.get("val_recall") or item.get("val_recall_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "F1-Macro", "Value": item["val_f1_macro"]})

    df_plot = pd.DataFrame(fold_data)

    # Create grouped bar plot
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    # Set premium cohesive color palette
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

    ax = sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_plot, 
        palette=colors,
        edgecolor="white",
        linewidth=1
    )

    plt.title("DistilBERT - Fold-Wise Validation Performance", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    plt.xlabel("Validation Folds", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.ylim(0.5, 1.0)  # Focus visual range on where the scores reside (50% to 100%)

    # Display numeric values on top of the bars
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")

    plt.legend(title="Metric", loc="lower right", frameon=True)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()

    # Save output plot to model folder
    save_path = results_path.parent / "3.10_distilbert_fold_wise_validation_summary.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: DistilBERT results not found at: {results_path}")


### 3.11  RoBERTa Model — Civic Agency (Dataset V2)

We fine-tune the pre-trained `roberta-base` model using 5-fold cross-validation. RoBERTa uses robust pre-training methods and shows well-balanced predictions across classes.

> **[!TIP]**
> * **Training Script:** [`scripts/roberta_train_civic_v2.py`](../scripts/roberta_train_civic_v2.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_civic_roberta_v2.py`](../scripts/submit_vertex_job_civic_roberta_v2.py)


In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../models/civic_bodies/dataset_v2/RoBERTa/oof_predictions_civic_roberta.joblib")

if oof_path.exists():
    data = joblib.load(oof_path)
    y_true = data["true"]
    y_pred = data["pred"]
    labels = data["labels"]

    # 1. Compute per-class binary accuracy and standard classification metrics
    class_data = []
    for label in sorted(labels):
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None
        )
        support = s_val[0]
        
        class_data.append({
            "Civic Agency": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Civic Agency")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # 2. Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("RoBERTa - Per-Agency Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Civic Agency & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("RoBERTa - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    # Save to model output folder and render
    save_path = oof_path.parent / "3.11_roberta_per_agency_metrics.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_civic_roberta.joblib not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/civic_bodies/dataset_v2/RoBERTa/results_roberta_civic.json")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Aggregate standard metrics per fold
        fold_data.append({"Fold": fold_label, "Metric": "Accuracy", "Value": item["val_accuracy"]})
        fold_data.append({"Fold": fold_label, "Metric": "Precision", "Value": item.get("val_precision") or item.get("val_precision_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "Recall", "Value": item.get("val_recall") or item.get("val_recall_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "F1-Macro", "Value": item["val_f1_macro"]})

    df_plot = pd.DataFrame(fold_data)

    # Create grouped bar plot
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    # Set premium cohesive color palette
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

    ax = sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_plot, 
        palette=colors,
        edgecolor="white",
        linewidth=1
    )

    plt.title("RoBERTa - Fold-Wise Validation Performance", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    plt.xlabel("Validation Folds", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.ylim(0.5, 1.0)  # Focus visual range on where the scores reside (50% to 100%)

    # Display numeric values on top of the bars
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")

    plt.legend(title="Metric", loc="lower right", frameon=True)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()

    # Save output plot to model folder
    save_path = results_path.parent / "3.11_roberta_fold_wise_validation_summary.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: RoBERTa results not found at: {results_path}")


### 3.12  DeBERTa V3 Model — Civic Agency (Dataset V2)

We fine-tune the pre-trained `microsoft/deberta-v3-base` model. DeBERTa v3 features disentangled attention and electra-style pretraining, yielding excellent overall accuracy.

> **[!TIP]**
> * **Training Script:** [`scripts/deberta_v3_train_civic_v2.py`](../scripts/deberta_v3_train_civic_v2.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_civic_deberta_v3.py`](../scripts/submit_vertex_job_civic_deberta_v3.py)


In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../data/processed/oof_predictions_civic_deberta_v3.joblib")

if oof_path.exists():
    data = joblib.load(oof_path)
    y_true = data["true"]
    y_pred = data["pred"]
    labels = data["labels"]

    # 1. Compute per-class binary accuracy and standard classification metrics
    class_data = []
    for label in sorted(labels):
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None
        )
        support = s_val[0]
        
        class_data.append({
            "Civic Agency": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Civic Agency")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # 2. Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("DeBERTa v3 - Per-Agency Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Civic Agency & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DeBERTa v3 - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    # Save to model output folder and render
    save_path = Path("../models/civic_bodies/dataset_v2/DeBERTa_v3/3.12_deberta_per_agency_metrics.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_civic_deberta_v3.joblib not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/civic_bodies/dataset_v2/DeBERTa_v3/results_deberta_v3_civic.json")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Aggregate standard metrics per fold
        fold_data.append({"Fold": fold_label, "Metric": "Accuracy", "Value": item["val_accuracy"]})
        fold_data.append({"Fold": fold_label, "Metric": "Precision", "Value": item.get("val_precision") or item.get("val_precision_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "Recall", "Value": item.get("val_recall") or item.get("val_recall_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "F1-Macro", "Value": item["val_f1_macro"]})

    df_plot = pd.DataFrame(fold_data)

    # Create grouped bar plot
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    # Set premium cohesive color palette
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

    ax = sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_plot, 
        palette=colors,
        edgecolor="white",
        linewidth=1
    )

    plt.title("DeBERTa v3 - Fold-Wise Validation Performance", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    plt.xlabel("Validation Folds", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.ylim(0.5, 1.0)  # Focus visual range on validation scores (50% to 100%)

    # Display numeric values on top of the bars
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")

    plt.legend(title="Metric", loc="lower right", frameon=True)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()

    # Save output plot to model folder
    save_path = results_path.parent / "3.12_deberta_fold_wise_validation_summary.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: DeBERTa v3 results not found at: {results_path}")


### 3.12  Civic Agency Model Comparison & Soft Stacking (Dataset V2)

To maximize prediction robustness, we combine the out-of-fold predictions using a ensembling layer. We compare standard Voting (Majority Vote) and Weighted Soft Voting (WSV).

> **[!TIP]**
> * **Probability Extraction Script:** [`scripts/extract_probs_civic.py`](../scripts/extract_probs_civic.py)
> * **Soft Stacking & Blending Script:** [`scripts/ensemble_soft_stacking.py`](../scripts/ensemble_soft_stacking.py)


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/civic_bodies/dataset_v2/ensemble_stacking_soft/results_soft_stacking.json")
save_path = Path("../models/civic_bodies/dataset_v2/ensemble_stacking_soft/3.13_ensemble_stacking_soft_comparison.png")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    # 1. Parse per-agency F1 scores
    df_agency_f1 = pd.DataFrame(data["per_agency_f1"]).T
    
    # 2. Parse overall metrics
    df_overall = pd.DataFrame(data["overall_metrics"]).T

    # Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(12, 9.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4.5, 2.0], hspace=0.35)

    # Top Plot: Heatmap of Per-Agency F1 Scores
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        df_agency_f1, 
        annot=True, 
        fmt=".4f", 
        cmap="RdYlGn", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("Ensemble & Model Comparison - Per-Agency F1-Scores", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("Civic Agency", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_ylabel("Approach", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table of Overall Metrics
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    # Prepare table data
    table_data = []
    for model_name, row in df_overall.iterrows():
        table_data.append([
            model_name,
            f"{row['Accuracy']:.2%}",
            f"{row['F1-Macro']:.4f}",
            f"{row['Precision-Macro']:.4f}",
            f"{row['Recall-Macro']:.4f}"
        ])

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Approach", "Accuracy (Overall)", "F1-Macro", "Precision-Macro", "Recall-Macro"],
        cellLoc="center",
        loc="center",
        colWidths=[0.25, 0.18, 0.18, 0.18, 0.18]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            # Highlight Weighted Soft Voting (WINNER) in soft green background
            model_name = table_data[row - 1][0]
            if model_name == "Weighted Soft Voting":
                cell.set_facecolor("#e8f5e9")  # Light green for winning ensemble
                cell.set_text_props(weight="bold", color="#1b5e20")
            elif row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
                cell.set_text_props(weight="normal")
            else:
                cell.set_facecolor("white")
                cell.set_text_props(weight="normal")

    ax_table.set_title("Overall Stacking & Base Model Comparison Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    # Save to model output folder and render
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: results_soft_stacking.json not found at: {results_path}")


### 3.14  Statistical Hypothesis Validation (McNemar's Test)

We perform a McNemar's Chi-Squared test to verify if the improvement of our Weighted Soft Voting (WSV) ensemble over the best single model (RoBERTa) is statistically significant.

> **[!TIP]**
> * **Hypothesis Testing Script:** [`tests/hypothesis_testing.py`](../tests/hypothesis_testing.py)


In [ ]:
# Explicit relative path escaping the notebook/ directory
results_path = Path("../models/civic_bodies/dataset_v2/ensemble_stacking_soft/hypothesis_test_results.json")

if results_path.exists():
    with open(results_path, "r") as f:
        hyp_results = json.load(f)
    
    mc = hyp_results["mcnemar"]
    
    print("==========================================================================")
    print("       McNemar's Chi-Squared Test: WSV Ensemble vs. Best Single Model     ")
    print("==========================================================================")
    print("Hypotheses:")
    print("  H0 (Null): The Weighted Soft Voting (WSV) ensemble and the best single")
    print("             model (RoBERTa) have equivalent error rates.")
    print("  H1 (Alt) : The WSV ensemble and the best single model have different")
    print("             error rates (i.e., the ensemble's improvement is genuine).")
    print("-" * 74)
    print(f"  McNemar Chi-Squared Statistic : {mc['statistic']:.4f}")
    print(f"  McNemar Test p-value         : {mc['p_value']:.4e}")
    print("-" * 74)
    if mc['p_value'] < 0.05:
        print("  Decision  : Reject H0 (p < 0.05)")
        print("  Conclusion: The performance improvement of the Weighted Soft Voting")
        print("              ensemble over the best single model is statistically")
        print("              highly significant and not due to random chance.")
    else:
        print("  Decision  : Fail to reject H0 (p >= 0.05)")
        print("  Conclusion: The performance difference between the ensemble and the")
        print("              best single model is not statistically significant.")
    print("==========================================================================")
else:
    print(f"Error: Hypothesis testing results file not found at: {results_path}")


## 4  Severity Classification & Reason Generation (Dataset V2)

We prioritize complaint response urgency (Critical, High, Medium, Low) using a dual sequence-to-sequence generation and pair classification architecture.

### Severity Evaluation Workflow
1. **T5-base model** generates a detailed severity reasoning.
2. **RoBERTa-base classifier** predicts the final urgency level based on the complaint text and generated reason concatenated together.

### 4.1  Trial 1: T5 Joint Severity Score & Reason Generation (Dataset V2)

Fine-tunes a joint sequence-to-sequence model to output the severity label and reasoning in a single pass.

> **[!TIP]**
> * **Joint Model Training Script:** [`scripts/T5_train_severity_v2.py`](../scripts/T5_train_severity_v2.py)
> * **Reasoning Model Training Script:** [`scripts/T5_base_train_reason.py`](../scripts/T5_base_train_reason.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_t5_base_reason.py`](../scripts/submit_vertex_job_t5_base_reason.py)


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/severity/dataset_v2/trial_1_t5/results_t5_severity_v2.json")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # 1. Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Split errors (MAE/RMSE) and scores (R2/ROUGE-L) for plotting
        fold_data.append({"Fold": fold_label, "Metric": "MAE", "Value": item["val_mae"], "Type": "Error (Lower is Better)"})
        fold_data.append({"Fold": fold_label, "Metric": "RMSE", "Value": item["val_rmse"], "Type": "Error (Lower is Better)"})
        fold_data.append({"Fold": fold_label, "Metric": "R2 Score", "Value": item["val_r2"], "Type": "Score (Higher is Better)"})
        fold_data.append({"Fold": fold_label, "Metric": "ROUGE-L", "Value": item["val_rouge_l"], "Type": "Score (Higher is Better)"})

    df_plot = pd.DataFrame(fold_data)

    # 2. Set up Seaborn styling and combined grid layout
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(14, 9.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[4.5, 1.5], hspace=0.35)

    # Left Plot: Validation Errors (MAE & RMSE) per Fold
    ax1 = fig.add_subplot(gs[0, 0])
    colors_error = ["#E53935", "#FFB300"]  # Red, Amber
    df_error = df_plot[df_plot["Type"] == "Error (Lower is Better)"]
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_error, 
        palette=colors_error,
        edgecolor="white",
        linewidth=1,
        ax=ax1
    )
    ax1.set_title("T5 Validation Errors (MAE & RMSE)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax1.set_xlabel("Folds", fontsize=11)
    ax1.set_ylabel("Error Value", fontsize=11)
    for container in ax1.containers:
        ax1.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax1.legend(title="Metric", loc="upper right")

    # Right Plot: Validation Scores (R2 & ROUGE-L) per Fold
    ax2 = fig.add_subplot(gs[0, 1])
    colors_score = ["#1E88E5", "#8E24AA"]  # Blue, Purple
    df_score = df_plot[df_plot["Type"] == "Score (Higher is Better)"]
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_score, 
        palette=colors_score,
        edgecolor="white",
        linewidth=1,
        ax=ax2
    )
    ax2.set_title("T5 Validation Scores (R² & ROUGE-L)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax2.set_xlabel("Folds", fontsize=11)
    ax2.set_ylabel("Score Value", fontsize=11)
    ax2.set_ylim(0.0, 1.0)
    for container in ax2.containers:
        ax2.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax2.legend(title="Metric", loc="lower right")

    # Bottom Spanned Plot: Styled Summary Table of Overall Average Metrics
    ax_table = fig.add_subplot(gs[1, :])
    ax_table.axis("off")

    table_data = [
        ["Average Validation Loss", f"{data.get('avg_val_loss', 0.0):.4f}"],
        ["Average Validation MAE (Error)", f"{data.get('avg_val_mae', 0.0):.4f}"],
        ["Average Validation RMSE (Error)", f"{data.get('avg_val_rmse', 0.0):.4f}"],
        ["Average R² Score (Fit)", f"{data.get('avg_val_r2', 0.0):.4f}"],
        ["Average ROUGE-L F1 (Text Quality)", f"{data.get('avg_val_rouge_l', 0.0):.4f}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric Name (Overall Average)", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.45, 0.45]
    )

    # Style properties for the summary table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("T5 Model - Overall Averaged Performance Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    sns.despine(left=True, bottom=True)
    plt.suptitle("T5 Joint Severity Model - Validation Performance Profile", fontsize=15, fontweight="bold", y=0.98, color="#0b3c5d")
    plt.tight_layout()

    # Save output plot
    save_path = results_path.parent / "4.1_t5_severity_validation_summary.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: T5 results not found at: {results_path}")


### 4.2  Trial 2: Severity Regression using Classical Regressors

Evaluates classical algorithms (LightGBM, XGBoost) fitted on TF-IDF vectors to predict a continuous severity score.

> **[!TIP]**
> * **Classical Regressor Training Script:** [`scripts/xgb_severity_regressor.py`](../scripts/xgb_severity_regressor.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_severity_xgb.py`](../scripts/submit_vertex_job_severity_xgb.py)


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/severity/dataset_v2/trial_2_regression/results_regression_severity.json")
save_path = Path("../models/severity/dataset_v2/trial_2_regression/4.2_classical_regression_comparison.png")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    # 1. Parse model metrics
    model_data = []
    for model_name, metrics in data["models"].items():
        model_data.append({
            "Model": model_name.replace("_", " ").title(),
            "Val R2": metrics["avg_val_r2"],
            "Val MAE": metrics["avg_val_mae"],
            "Val RMSE": metrics["avg_val_rmse"],
            "Train R2": metrics["avg_train_r2"],
            "Train MAE": metrics["avg_train_mae"],
            "Train RMSE": metrics["avg_train_rmse"]
        })

    df_models = pd.DataFrame(model_data)

    # Create combined figure canvas
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[4.5, 2.0], hspace=0.35)

    # Top Left Plot: R2 Score Comparison
    ax1 = fig.add_subplot(gs[0, 0])
    sns.barplot(
        x="Model",
        y="Val R2",
        hue="Model",
        data=df_models,
        palette="Blues_r",
        edgecolor="white",
        linewidth=1,
        legend=False,
        ax=ax1
    )
    ax1.set_title("Validation R² Score Comparison (Higher is Better)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax1.set_xlabel("Classical Regressor Model", fontsize=11)
    ax1.set_ylabel("R² Score", fontsize=11)
    ax1.set_ylim(0.0, 0.7)
    for container in ax1.containers:
        ax1.bar_label(container, fmt="%.4f", padding=3, fontsize=9, fontweight="semibold")

    # Top Right Plot: MAE & RMSE Comparison
    ax2 = fig.add_subplot(gs[0, 1])
    # Reshape for grouped bar plot
    df_errors = df_models.melt(
        id_vars=["Model"],
        value_vars=["Val MAE", "Val RMSE"],
        var_name="Metric",
        value_name="Error Value"
    )
    df_errors["Metric"] = df_errors["Metric"].apply(lambda x: x.replace("Val ", ""))
    
    sns.barplot(
        x="Model",
        y="Error Value",
        hue="Metric",
        data=df_errors,
        palette=["#E53935", "#FFB300"], # Red, Amber
        edgecolor="white",
        linewidth=1,
        ax=ax2
    )
    ax2.set_title("Validation Errors Comparison (Lower is Better)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax2.set_xlabel("Classical Regressor Model", fontsize=11)
    ax2.set_ylabel("Error Value", fontsize=11)
    for container in ax2.containers:
        ax2.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax2.legend(title="Metric", loc="upper right")

    # Bottom Spanned Plot: Styled Table
    ax_table = fig.add_subplot(gs[1, :])
    ax_table.axis("off")

    table_data = []
    for _, row in df_models.iterrows():
        table_data.append([
            row["Model"],
            f"{row['Val R2']:.4f}",
            f"{row['Val MAE']:.2f}",
            f"{row['Val RMSE']:.2f}",
            f"{row['Train R2']:.4f}",
            f"{row['Train MAE']:.2f}",
            f"{row['Train RMSE']:.2f}"
        ])

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Model / Regressor", "Validation R²", "Validation MAE", "Validation RMSE", "Train R²", "Train MAE", "Train RMSE"],
        cellLoc="center",
        loc="center",
        colWidths=[0.22, 0.13, 0.13, 0.13, 0.13, 0.13, 0.13]
    )

    # Style the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            model_name = table_data[row - 1][0]
            # Highlight LightGBM (WINNER) in soft green
            if model_name == "Lightgbm":
                cell.set_facecolor("#e8f5e9")
                cell.set_text_props(weight="bold", color="#1b5e20")
            elif row % 2 == 0:
                cell.set_facecolor("#f9f9f9")
                cell.set_text_props(weight="normal")
            else:
                cell.set_facecolor("white")
                cell.set_text_props(weight="normal")

    ax_table.set_title("Classical Regressors - Performance Profile Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    sns.despine(left=True, bottom=True)
    plt.suptitle("Trial 2 Classical Regressors - Validation Performance Comparison", fontsize=15, fontweight="bold", y=0.98, color="#0b3c5d")
    
    # Adjust layout manually to prevent tight_layout warning with table
    plt.subplots_adjust(top=0.90, bottom=0.05, left=0.05, right=0.95, hspace=0.35)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: results_regression_severity.json not found at: {results_path}")


### 4.3  Trial 3: DistilBERT Regressor

Fine-tunes a pre-trained `distilbert-base-uncased` with a linear regression head targeting continuous severity scores.

> **[!TIP]**
> * **DistilBERT Regressor Script:** [`scripts/distilbert_severity_regressor.py`](../scripts/distilbert_severity_regressor.py)
> * **Vertex AI Job Submission:** [`scripts/submit_vertex_job_severity_distilbert_reg.py`](../scripts/submit_vertex_job_severity_distilbert_reg.py)


In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../models/severity/dataset_v2/trial_3_distilBERT/oof_predictions_distilbert_reg.json")
save_path = Path("../models/severity/dataset_v2/trial_3_distilBERT/4.3_distilbert_severity_per_class_metrics.png")

if oof_path.exists():
    with open(oof_path, "r") as f:
        oof_data = json.load(f)

    # Threshold mapping to reconstruct classes from continuous scores
    def get_severity(score):
        if score >= 90:
            return "Critical"
        elif score >= 80:
            return "High"
        elif score >= 50:
            return "Medium"
        elif score >= 1:
            return "Low"
        else:
            return "Non-Grievance"

    # 1. Reconstruct and bin all true and predicted scores across folds
    y_true_list = []
    y_pred_list = []
    for fold_item in oof_data["folds"]:
        y_true_list.extend([get_severity(s) for s in fold_item["true_scores"]])
        y_pred_list.extend([get_severity(s) for s in fold_item["pred_scores"]])

    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    labels_order = ["Non-Grievance", "Low", "Medium", "High", "Critical"]

    # 2. Compute per-class classification metrics
    class_data = []
    for label in labels_order:
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None, zero_division=0
        )
        support = s_val[0]
        
        class_data.append({
            "Severity Class": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Severity Class")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("DistilBERT Regressor (Binned) - Per-Class Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Severity Class & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall Binned)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DistilBERT Regressor (Binned) - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_distilbert_reg.json not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/severity/dataset_v2/trial_3_distilBERT/results_distilbert_regressor.json")
save_path = Path("../models/severity/dataset_v2/trial_3_distilBERT/4.3_distilbert_severity_validation_summary.png")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Split errors (MAE/RMSE) and scores (R2) for plotting
        fold_data.append({"Fold": fold_label, "Metric": "MAE", "Value": item["val_mae"], "Type": "Error (Lower is Better)"})
        fold_data.append({"Fold": fold_label, "Metric": "RMSE", "Value": item["val_rmse"], "Type": "Error (Lower is Better)"})
        fold_data.append({"Fold": fold_label, "Metric": "R2 Score", "Value": item["val_r2"], "Type": "Score (Higher is Better)"})

    df_plot = pd.DataFrame(fold_data)

    # Set up Seaborn styling and combined grid layout
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(14, 9.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[4.5, 1.5], hspace=0.35)

    # Left Plot: Validation Errors (MAE & RMSE) per Fold
    ax1 = fig.add_subplot(gs[0, 0])
    colors_error = ["#E53935", "#FFB300"]  # Red, Amber
    df_error = df_plot[df_plot["Type"] == "Error (Lower is Better)"]
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_error, 
        palette=colors_error,
        edgecolor="white",
        linewidth=1,
        ax=ax1
    )
    ax1.set_title("DistilBERT Validation Errors (MAE & RMSE)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax1.set_xlabel("Folds", fontsize=11)
    ax1.set_ylabel("Error Value", fontsize=11)
    for container in ax1.containers:
        ax1.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax1.legend(title="Metric", loc="upper right")

    # Right Plot: Validation Scores (R2) per Fold
    ax2 = fig.add_subplot(gs[0, 1])
    colors_score = ["#1E88E5"]  # Blue
    df_score = df_plot[df_plot["Type"] == "Score (Higher is Better)"]
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        hue_order=["R2 Score"],
        data=df_score, 
        palette=colors_score,
        edgecolor="white",
        linewidth=1,
        ax=ax2
    )
    ax2.set_title("DistilBERT Validation Scores (R²)", fontsize=13, fontweight="bold", pad=12, color="#0b3c5d")
    ax2.set_xlabel("Folds", fontsize=11)
    ax2.set_ylabel("R² Score Value", fontsize=11)
    ax2.set_ylim(0.0, 1.0)
    for container in ax2.containers:
        ax2.bar_label(container, fmt="%.4f", padding=3, fontsize=9, fontweight="semibold")
    ax2.legend(title="Metric", loc="lower right")

    # Bottom Spanned Plot: Styled Summary Table of Overall Average Metrics
    ax_table = fig.add_subplot(gs[1, :])
    ax_table.axis("off")

    table_data = [
        ["Average Validation Loss", f"{data.get('avg_val_loss', 0.0):.4f}"],
        ["Average Validation MAE (Error)", f"{data.get('avg_val_mae', 0.0):.4f}"],
        ["Average Validation RMSE (Error)", f"{data.get('avg_val_rmse', 0.0):.4f}"],
        ["Average R² Score (Fit)", f"{data.get('avg_val_r2', 0.0):.4f}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric Name (Overall Average)", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.45, 0.45]
    )

    # Style properties for the summary table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DistilBERT Regressor - Overall Averaged Performance Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    sns.despine(left=True, bottom=True)
    plt.suptitle("DistilBERT Regressor - Validation Performance Profile", fontsize=15, fontweight="bold", y=0.98, color="#0b3c5d")
    
    # Adjust layout manually to prevent tight_layout warning with table
    plt.subplots_adjust(top=0.90, bottom=0.05, left=0.05, right=0.95, hspace=0.35)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: results_distilbert_regressor.json not found at: {results_path}")


### 4.4  Trial 4: RoBERTa Severity Classifier (Concatenated Text Pairs) — WINNER

We train classifiers on concatenated sentence pairs: `[complaint text] [SEP] [T5-generated severity reason]`. The RoBERTa classifier achieves best-in-class results and is selected for production.

> **[!TIP]**
> * **RoBERTa Classifier Script:** [`scripts/roberta_classifier_severity_v2.py`](../scripts/roberta_classifier_severity_v2.py)
> * **RoBERTa Vertex AI Submission:** [`scripts/submit_vertex_job_severity_roberta_v2.py`](../scripts/submit_vertex_job_severity_roberta_v2.py)


In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../models/severity/dataset_v2/trial_5_roberta_classifier/oof_predictions_roberta_classifier_v2.json")
save_path = Path("../models/severity/dataset_v2/trial_5_roberta_classifier/4.5_roberta_severity_per_class_metrics.png")

if oof_path.exists():
    with open(oof_path, "r") as f:
        oof_data = json.load(f)

    # 1. Reconstruct all true and predicted labels across folds
    y_true_int = []
    y_pred_int = []
    for fold_item in oof_data["folds"]:
        y_true_int.extend(fold_item["true_labels"])
        y_pred_int.extend(fold_item["pred_labels"])

    y_true_int = np.array(y_true_int)
    y_pred_int = np.array(y_pred_int)

    # Label mapping (0: Non-Grievance, 1: Low, 2: Medium, 3: High, 4: Critical)
    label_map = {
        0: "Non-Grievance",
        1: "Low",
        2: "Medium",
        3: "High",
        4: "Critical"
    }

    y_true = np.array([label_map[val] for val in y_true_int])
    y_pred = np.array([label_map[val] for val in y_pred_int])
    labels_order = ["Non-Grievance", "Low", "Medium", "High", "Critical"]

    # 2. Compute per-class classification metrics
    class_data = []
    for label in labels_order:
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None, zero_division=0
        )
        support = s_val[0]
        
        class_data.append({
            "Severity Class": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Severity Class")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("RoBERTa Severity Classifier - Per-Class Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Severity Class & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("RoBERTa Severity Classifier - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_roberta_classifier_v2.json not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/severity/dataset_v2/trial_5_roberta_classifier/results_roberta_classifier_v2.json")
save_path = Path("../models/severity/dataset_v2/trial_5_roberta_classifier/4.5_roberta_severity_validation_summary.png")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Aggregate standard classification metrics per fold
        fold_data.append({"Fold": fold_label, "Metric": "Accuracy", "Value": item["val_accuracy"]})
        fold_data.append({"Fold": fold_label, "Metric": "Precision", "Value": item.get("val_precision_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "Recall", "Value": item.get("val_recall_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "F1-Macro", "Value": item["val_f1_macro"]})

    df_plot = pd.DataFrame(fold_data)

    # Set up Seaborn styling and combined grid layout
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(14, 9.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4.5, 1.8], hspace=0.35)

    # Top Plot: Grouped Bar Chart for Fold-Wise Metrics
    ax_bar = fig.add_subplot(gs[0])
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]  # Blue, Green, Orange, Purple
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_plot, 
        palette=colors,
        edgecolor="white",
        linewidth=1,
        ax=ax_bar
    )
    ax_bar.set_title("RoBERTa Severity Classifier - Fold-Wise Validation Performance", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_bar.set_xlabel("Validation Folds", fontsize=12)
    ax_bar.set_ylabel("Score", fontsize=12)
    ax_bar.set_ylim(0.4, 1.0)  # Focus visual range
    for container in ax_bar.containers:
        ax_bar.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax_bar.legend(title="Metric", loc="lower right", frameon=True)

    # Bottom Spanned Plot: Styled Summary Table of Overall Average Metrics
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Average Validation Loss", f"{data.get('avg_val_loss', 0.0):.4f}"],
        ["Average Validation Accuracy", f"{data.get('avg_val_acc', 0.0):.2%}"],
        ["Average Validation F1-Macro", f"{data.get('avg_val_f1', 0.0):.4f}"],
        ["Average Validation F1-Weighted", f"{data.get('avg_val_f1_weighted', 0.0):.4f}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric Name (Overall Average)", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.45, 0.45]
    )

    # Style properties for the summary table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("RoBERTa Severity Classifier - Overall Averaged Performance Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    sns.despine(left=True, bottom=True)
    plt.suptitle("RoBERTa Severity Classifier (Concatenated Text Pairs) - Validation Profile", fontsize=15, fontweight="bold", y=0.98, color="#0b3c5d")
    
    # Adjust layout manually to prevent tight_layout warning with table
    plt.subplots_adjust(top=0.90, bottom=0.05, left=0.05, right=0.95, hspace=0.35)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: results_roberta_classifier_v2.json not found at: {results_path}")


### 4.5  Trial 5: DeBERTa v3 Severity Classifier:
We train classifiers on concatenated sentence pairs: `[complaint text] [SEP] [T5-generated severity reason]`. The RoBERTa classifier achieves best-in-class results and is selected for production.

> * **DeBERTa v3 Classifier Script:** [`scripts/deberta_v3_classifier_severity_v2.py`](../scripts/deberta_v3_classifier_severity_v2.py)
> * **DeBERTa v3 Vertex Submission:** [`scripts/submit_vertex_job_severity_deberta_v3.py`](../scripts/submit_vertex_job_severity_deberta_v3.py)

In [ ]:
# Explicit relative path from notebook/ folder
oof_path = Path("../models/severity/dataset_v2/deberta_v3_classifier/oof_predictions_deberta_v3_classifier_v2.json")
save_path = Path("../models/severity/dataset_v2/deberta_v3_classifier/4.4_deberta_severity_per_class_metrics.png")

if oof_path.exists():
    with open(oof_path, "r") as f:
        oof_data = json.load(f)

    # 1. Reconstruct all true and predicted labels across folds
    y_true_int = []
    y_pred_int = []
    for fold_item in oof_data["folds"]:
        y_true_int.extend(fold_item["true_labels"])
        y_pred_int.extend(fold_item["pred_labels"])

    y_true_int = np.array(y_true_int)
    y_pred_int = np.array(y_pred_int)

    # Label mapping (0: Non-Grievance, 1: Low, 2: Medium, 3: High, 4: Critical)
    label_map = {
        0: "Non-Grievance",
        1: "Low",
        2: "Medium",
        3: "High",
        4: "Critical"
    }

    y_true = np.array([label_map[val] for val in y_true_int])
    y_pred = np.array([label_map[val] for val in y_pred_int])
    labels_order = ["Non-Grievance", "Low", "Medium", "High", "Critical"]

    # 2. Compute per-class classification metrics
    class_data = []
    for label in labels_order:
        bin_true = (y_true == label)
        bin_pred = (y_pred == label)
        acc = accuracy_score(bin_true, bin_pred)
        
        p_val, r_val, f_val, s_val = precision_recall_fscore_support(
            y_true, y_pred, labels=[label], average=None, zero_division=0
        )
        support = s_val[0]
        
        class_data.append({
            "Severity Class": f"{label} (n={support:,})",
            "Accuracy (One-vs-All)": acc,
            "Precision": p_val[0],
            "Recall": r_val[0],
            "F1-Score": f_val[0]
        })

    df_metrics = pd.DataFrame(class_data)
    heatmap_data = df_metrics.set_index("Severity Class")[
        ["Accuracy (One-vs-All)", "Precision", "Recall", "F1-Score"]
    ]

    # Compute overall summary statistics
    acc_overall = accuracy_score(y_true, y_pred)
    macro_p = heatmap_data["Precision"].mean()
    macro_r = heatmap_data["Recall"].mean()
    macro_f1 = heatmap_data["F1-Score"].mean()
    total_support = len(y_true)

    # 3. Create combined figure canvas (Heatmap + Styled Table)
    fig = plt.figure(figsize=(10, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.3)

    # Top Plot: Heatmap
    ax_heatmap = fig.add_subplot(gs[0])
    sns.heatmap(
        heatmap_data, 
        annot=True, 
        fmt=".4f", 
        cmap="Blues", 
        cbar=False, 
        linewidths=0.5,
        ax=ax_heatmap,
        annot_kws={"size": 10, "weight": "bold", "color": "black"}
    )
    ax_heatmap.set_title("DeBERTa v3 Severity Classifier - Per-Class Metrics Heatmap", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("Severity Class & Class Support (n)", fontsize=11, fontweight="bold", labelpad=10)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), rotation=0, fontweight="bold", fontsize=9)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), rotation=0, fontweight="bold", fontsize=9)

    # Bottom Plot: Styled Summary Table
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Accuracy (Overall)", f"{acc_overall:.2%}"],
        ["Macro Precision", f"{macro_p:.4f}"],
        ["Macro Recall", f"{macro_r:.4f}"],
        ["Macro F1-Score", f"{macro_f1:.4f}"],
        ["Total Support", f"{total_support:,}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.4, 0.4]
    )

    # Apply style properties to the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#00b074")  # Emerald green header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DeBERTa v3 Severity Classifier - Overall Summary Performance", fontsize=12, fontweight="bold", color="#00b074", pad=5)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: oof_predictions_deberta_v3_classifier_v2.json not found at: {oof_path}")


In [ ]:
# Explicit relative path from notebook/ folder
results_path = Path("../models/severity/dataset_v2/deberta_v3_classifier/results_deberta_v3_classifier_v2.json")
save_path = Path("../models/severity/dataset_v2/deberta_v3_classifier/4.4_deberta_severity_validation_summary.png")

if results_path.exists():
    with open(results_path, "r") as f:
        data = json.load(f)

    per_fold = data.get("per_fold_metrics", [])

    # Format the fold-wise metrics into a long-form DataFrame for Seaborn
    fold_data = []
    for item in per_fold:
        fold_idx = item["fold"] if item["fold"] is not None else 0
        fold_label = f"Fold {fold_idx}"
        
        # Aggregate standard classification metrics per fold
        fold_data.append({"Fold": fold_label, "Metric": "Accuracy", "Value": item["val_accuracy"]})
        fold_data.append({"Fold": fold_label, "Metric": "Precision", "Value": item.get("val_precision_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "Recall", "Value": item.get("val_recall_macro") or 0.0})
        fold_data.append({"Fold": fold_label, "Metric": "F1-Macro", "Value": item["val_f1_macro"]})

    df_plot = pd.DataFrame(fold_data)

    # Set up Seaborn styling and combined grid layout
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(14, 9.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4.5, 1.8], hspace=0.35)

    # Top Plot: Grouped Bar Chart for Fold-Wise Metrics
    ax_bar = fig.add_subplot(gs[0])
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]  # Blue, Green, Orange, Purple
    sns.barplot(
        x="Fold", 
        y="Value", 
        hue="Metric", 
        data=df_plot, 
        palette=colors,
        edgecolor="white",
        linewidth=1,
        ax=ax_bar
    )
    ax_bar.set_title("DeBERTa v3 Severity Classifier - Fold-Wise Validation Performance", fontsize=14, fontweight="bold", pad=15, color="#0b3c5d")
    ax_bar.set_xlabel("Validation Folds", fontsize=12)
    ax_bar.set_ylabel("Score", fontsize=12)
    ax_bar.set_ylim(0.4, 1.0)  # Focus visual range
    for container in ax_bar.containers:
        ax_bar.bar_label(container, fmt="%.2f", padding=3, fontsize=9, fontweight="semibold")
    ax_bar.legend(title="Metric", loc="lower right", frameon=True)

    # Bottom Spanned Plot: Styled Summary Table of Overall Average Metrics
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis("off")

    table_data = [
        ["Average Validation Loss", f"{data.get('avg_val_loss', 0.0):.4f}"],
        ["Average Validation Accuracy", f"{data.get('avg_val_acc', 0.0):.2%}"],
        ["Average Validation F1-Macro", f"{data.get('avg_val_f1', 0.0):.4f}"],
        ["Average Validation F1-Weighted", f"{data.get('avg_val_f1_weighted', 0.0):.4f}"]
    ]

    table = ax_table.table(
        cellText=table_data,
        colLabels=["Metric Name (Overall Average)", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.45, 0.45]
    )

    # Style properties for the summary table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.4)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#1e3d59")  # Dark blue header
            cell.set_edgecolor("#e0e0e0")
        else:
            cell.set_edgecolor("#e0e0e0")
            if row % 2 == 0:
                cell.set_facecolor("#f9f9f9")  # Alternating row coloring
            else:
                cell.set_facecolor("white")
            cell.set_text_props(weight="normal")

    ax_table.set_title("DeBERTa v3 Severity Classifier - Overall Averaged Performance Summary", fontsize=12, fontweight="bold", color="#1e3d59", pad=5)

    sns.despine(left=True, bottom=True)
    plt.suptitle("DeBERTa v3 Severity Classifier - Validation Profile", fontsize=15, fontweight="bold", y=0.98, color="#0b3c5d")
    
    # Adjust layout manually to prevent tight_layout warning with table
    plt.subplots_adjust(top=0.90, bottom=0.05, left=0.05, right=0.95, hspace=0.35)

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    
else:
    print(f"Error: results_deberta_v3_classifier_v2.json not found at: {results_path}")


## 5  Summary & Next Steps

### Key Accomplishments
1. **Blended Civic Ensemble**: Weighted Soft Voting across DistilBERT, RoBERTa, and DeBERTa v3 achieves **93.06% accuracy** and is statistically validated with a McNemar Chi-Squared test.
2. **T5 Explainer & RoBERTa Classifier**: Sequential pair classification solves class imbalance and leverages chain-of-thought severity reasons, yielding **76.11% accuracy**.
3. **Production API & SPA**: Serves real-time routing predictions with a beautiful glassmorphic Tailwind dashboard, live Chart.js visualization shares, and automated mock-based unit testing.

### Production Setup
- **Deployment**: Deployed on Hugging Face Spaces using Docker.
- **Code Base**: Re-aligned and clean modular scripts in `scripts/` directory.